### Model 1 ICD + CPT

In [1]:
# ============================================================
# JOINT ICD + CPT CODE PREDICTION USING BIOCLINICALBERT
# Folder-based split loading
#
# Expected folder structure:
# data/
#   train/
#       train1.jsonl
#       train2.jsonl
#       train3.jsonl
#       train4.jsonl
#   val/
#       val.jsonl
#   test/
#       test.jsonl
#
# Outputs saved to:
#   ./model1_joint_icd_cpt_output
# ============================================================


# ============================================================
# IMPORTS
# ============================================================
import os
import re
import json
import copy
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from safetensors.torch import save_file

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    # Keep same folders and output directory
    "train_dir": "data/train",
    "val_dir": "data/val",
    "test_dir": "data/test",
    "output_dir": "./model1_joint_icd_cpt_output",

    # Model
    "pretrained_model": "emilyalsentzer/Bio_ClinicalBERT",
    "max_length": 512,

    # Training
    # Your log showed best validation behavior around epoch 5.
    # So we use 7 epochs with patience 2.
    "batch_size": 8,
    "grad_accum_steps": 2,
    "epochs": 7,
    "patience": 2,

    "encoder_lr": 1.5e-5,
    "classifier_lr": 8e-5,
    "weight_decay": 0.02,
    "warmup_ratio": 0.10,
    "dropout": 0.25,
    "grad_clip": 1.0,

    # Label filtering
    "min_icd_label_freq": 2,
    "min_cpt_label_freq": 2,
    "top_k_icd_labels": 50,
    "top_k_cpt_labels": 50,

    # ICD needs stronger but controlled recall
    "use_icd_pos_weight": True,
    "icd_pos_weight_clip": 1.5,

    # CPT already passed in your previous run
    "use_cpt_pos_weight": False,
    "cpt_pos_weight_clip": 1.0,

    "icd_loss_weight": 1.35,
    "cpt_loss_weight": 0.90,

    # Fallback thresholds only
    "icd_threshold": 0.30,
    "cpt_threshold": 0.35,

    # ICD validation threshold grid
    "icd_threshold_grid": [
        0.20, 0.22, 0.24, 0.26, 0.28,
        0.30, 0.32, 0.34, 0.36, 0.38, 0.40
    ],

    # CPT should stay controlled
    "cpt_threshold_grid": [
        0.32, 0.34, 0.36, 0.38, 0.40, 0.45, 0.50
    ],

    # ICD needs more prediction capacity
    "icd_topk_grid": [8, 10, 12, 15],
    "icd_max_predictions_grid": [8, 10, 12, 15],

    # CPT already passes, keep it tighter
    "cpt_topk_grid": [4, 5, 6],
    "cpt_max_predictions_grid": [4, 5, 6],

    "icd_min_predictions": 1,
    "cpt_min_predictions": 0,

    # Do not punish ICD precision too strongly because recall is the problem
    "icd_min_val_precision": 0.32,
    "cpt_min_val_precision": 0.40,

    # Checkpoint selection strongly favors ICD while ensuring CPT does not collapse
    "selection_icd_weight": 0.90,
    "selection_cpt_weight": 0.10,

    # ICD-focused text construction
    "max_note_chars": 14000,
    "max_lab_events": 80,

    # System
    "num_workers": 0,
    "fp16": True,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("All outputs will be saved to:", CONFIG["output_dir"])


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])


# ============================================================
# HELPERS
# ============================================================
def natural_sort_key(path_obj):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path_obj.name)
    ]


def safe_str(x):
    if x is None:
        return ""
    return str(x)


def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def make_state_dict_safe_for_saving(state_dict):
    safe_state_dict = {}

    for k, v in state_dict.items():
        if isinstance(v, torch.Tensor):
            safe_state_dict[k] = v.detach().cpu().contiguous()
        else:
            safe_state_dict[k] = v

    return safe_state_dict


# ============================================================
# FILE LOADING
# ============================================================
def get_jsonl_files(folder_path: str):
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    files = sorted(folder.glob("*.jsonl"), key=natural_sort_key)

    if len(files) == 0:
        raise FileNotFoundError(f"No .jsonl files found in: {folder_path}")

    return files


def load_jsonl_file(file_path: Path) -> pd.DataFrame:
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON in {file_path} at line {line_num}: {e}"
                ) from e

    return pd.DataFrame(rows)


def load_split_from_folder(folder_path: str, split_name: str) -> pd.DataFrame:
    files = get_jsonl_files(folder_path)

    print(f"\nLoading {split_name} files from: {folder_path}")
    for f in files:
        print(" -", f.name)

    dfs = []

    for file_path in files:
        df = load_jsonl_file(file_path)
        print(f"Loaded {file_path.name} -> shape: {df.shape}")
        dfs.append(df)

    split_df = pd.concat(dfs, ignore_index=True)
    print(f"Combined {split_name} shape: {split_df.shape}")

    return split_df


# ============================================================
# EVENT SERIALIZATION
# ============================================================
def format_lab_event(event: dict) -> str:
    value = event.get("value", {}) or {}

    label = safe_str(value.get("label", "unknown_lab")).strip()
    valuenum = value.get("valuenum", None)
    valueuom = safe_str(value.get("valueuom", "")).strip()
    abnormal = value.get("is_abnormal", None)
    rel_time = safe_float(event.get("relative_time_hrs", None))

    abnormal_text = ""
    if abnormal is True:
        abnormal_text = " abnormal"
    elif abnormal is False:
        abnormal_text = " normal"

    time_text = f" at {rel_time:.2f} hours" if rel_time is not None else ""

    if valuenum is None:
        return f"Lab {label}{time_text}{abnormal_text}."

    return f"Lab {label} {valuenum} {valueuom}{time_text}{abnormal_text}."


def format_text_event(event: dict, event_type: str) -> str:
    text = safe_str(event.get("text", "")).strip()
    text = " ".join(text.split())

    rel_time = safe_float(event.get("relative_time_hrs", None))
    time_text = f" at {rel_time:.2f} hours" if rel_time is not None else ""

    return f"{event_type.capitalize()} note{time_text}: {text}"


def format_generic_event(event: dict) -> str:
    event_type = safe_str(event.get("event_type", "unknown")).strip().lower()
    rel_time = safe_float(event.get("relative_time_hrs", None))
    time_text = f" at {rel_time:.2f} hours" if rel_time is not None else ""

    if "text" in event and event["text"] is not None:
        return format_text_event(event, event_type)

    value = event.get("value", None)

    if isinstance(value, dict):
        pairs = []

        for k, v in value.items():
            if v is not None:
                pairs.append(f"{k}={v}")

        joined = ", ".join(pairs)

        if joined:
            return f"{event_type.capitalize()} event{time_text}: {joined}"

    return f"{event_type.capitalize()} event{time_text}."


def serialize_events_to_text(events) -> str:
    """
    ICD-focused serialization.

    Important change:
    The old version sorted everything by time.
    That can waste the first 512 tokens on labs or less useful early events.

    This version places diagnosis-heavy text first:
        discharge -> physician -> radiology -> nursing -> other -> labs

    This is the most important change for Model 1 ICD improvement.
    """
    if not isinstance(events, list):
        return ""

    priority = {
        "discharge": 0,
        "physician": 1,
        "radiology": 2,
        "nursing": 3,
        "lab": 5
    }

    def sort_key(event):
        event_type = safe_str(event.get("event_type", "")).lower().strip()
        rel = safe_float(event.get("relative_time_hrs", None))
        rel_val = float("inf") if rel is None else rel
        return (priority.get(event_type, 4), rel_val)

    events_sorted = sorted(events, key=sort_key)

    note_lines = []
    other_lines = []
    lab_lines = []

    lab_count = 0

    for event in events_sorted:
        event_type = safe_str(event.get("event_type", "")).lower().strip()

        if event_type == "lab":
            if lab_count < CONFIG.get("max_lab_events", 80):
                lab_lines.append(format_lab_event(event))
                lab_count += 1

        elif event_type in {"discharge", "physician", "radiology", "nursing"}:
            note_lines.append(format_text_event(event, event_type))

        else:
            other_lines.append(format_generic_event(event))

    # Diagnosis-heavy notes first, labs last
    full_text = " ".join(note_lines + other_lines + lab_lines)
    full_text = " ".join(full_text.split())

    max_chars = CONFIG.get("max_note_chars", 14000)
    return full_text[:max_chars].strip()


# ============================================================
# LABEL HANDLING
# ============================================================
def normalize_label_list(x):
    if x is None:
        return []

    if isinstance(x, list):
        out = []

        for item in x:
            if item is None:
                continue

            if isinstance(item, dict):
                for key in ["code", "label", "value", "name"]:
                    if key in item and item[key] is not None:
                        val = str(item[key]).strip()

                        if val:
                            out.append(val)

                        break

            else:
                val = str(item).strip()

                if val:
                    out.append(val)

        return out

    if isinstance(x, str):
        x = x.strip()

        if not x:
            return []

        for sep in [",", ";", "|"]:
            if sep in x:
                return [part.strip() for part in x.split(sep) if part.strip()]

        return [x]

    val = str(x).strip()
    return [val] if val else []


def extract_icd_labels(label_obj):
    if not isinstance(label_obj, dict):
        return []

    return normalize_label_list(label_obj.get("icd10", []))


def extract_cpt_labels(label_obj):
    if not isinstance(label_obj, dict):
        return []

    return normalize_label_list(label_obj.get("cpt", []))


def get_kept_labels_from_train(df_train: pd.DataFrame, label_col: str, min_freq: int, top_k):
    counter = Counter()

    for labels in df_train[label_col]:
        counter.update(labels)

    kept = [lab for lab, freq in counter.items() if freq >= min_freq]
    kept = sorted(kept, key=lambda x: counter[x], reverse=True)

    if top_k is not None:
        kept = kept[:top_k]

    return kept


def apply_label_filter(df: pd.DataFrame, label_col: str, kept_labels):
    kept_set = set(kept_labels)
    df = df.copy()

    df[label_col] = df[label_col].apply(
        lambda labs: [x for x in labs if x in kept_set]
    )

    return df


# ============================================================
# DATA PREP
# ============================================================
def preprocess_split(df_raw: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if "events" not in df_raw.columns:
        raise ValueError(f"{split_name} dataset must contain 'events' column.")

    if "labels" not in df_raw.columns:
        raise ValueError(f"{split_name} dataset must contain 'labels' column.")

    df = df_raw.copy()

    df["text"] = df["events"].apply(serialize_events_to_text)
    df["icd_labels"] = df["labels"].apply(extract_icd_labels)
    df["cpt_labels"] = df["labels"].apply(extract_cpt_labels)

    df["text"] = df["text"].fillna("").astype(str)
    df = df[df["text"].str.len() > 0].copy()

    df = df[
        (df["icd_labels"].map(len) > 0) |
        (df["cpt_labels"].map(len) > 0)
    ].copy()

    if len(df) == 0:
        raise ValueError(f"No usable rows left in {split_name} after preprocessing.")

    return df.reset_index(drop=True)


def build_datasets(train_raw: pd.DataFrame, val_raw: pd.DataFrame, test_raw: pd.DataFrame):
    train_df = preprocess_split(train_raw, "train")
    val_df = preprocess_split(val_raw, "val")
    test_df = preprocess_split(test_raw, "test")

    kept_icd = get_kept_labels_from_train(
        train_df,
        label_col="icd_labels",
        min_freq=CONFIG["min_icd_label_freq"],
        top_k=CONFIG["top_k_icd_labels"]
    )

    kept_cpt = get_kept_labels_from_train(
        train_df,
        label_col="cpt_labels",
        min_freq=CONFIG["min_cpt_label_freq"],
        top_k=CONFIG["top_k_cpt_labels"]
    )

    if len(kept_icd) == 0:
        raise ValueError(
            "No ICD labels left after filtering train data. "
            "Try reducing min_icd_label_freq or setting top_k_icd_labels=None."
        )

    if len(kept_cpt) == 0:
        raise ValueError(
            "No CPT labels left after filtering train data. "
            "Try reducing min_cpt_label_freq or setting top_k_cpt_labels=None."
        )

    train_df = apply_label_filter(train_df, "icd_labels", kept_icd)
    val_df = apply_label_filter(val_df, "icd_labels", kept_icd)
    test_df = apply_label_filter(test_df, "icd_labels", kept_icd)

    train_df = apply_label_filter(train_df, "cpt_labels", kept_cpt)
    val_df = apply_label_filter(val_df, "cpt_labels", kept_cpt)
    test_df = apply_label_filter(test_df, "cpt_labels", kept_cpt)

    train_df = train_df[
        (train_df["icd_labels"].map(len) > 0) |
        (train_df["cpt_labels"].map(len) > 0)
    ].copy()

    val_df = val_df[
        (val_df["icd_labels"].map(len) > 0) |
        (val_df["cpt_labels"].map(len) > 0)
    ].copy()

    test_df = test_df[
        (test_df["icd_labels"].map(len) > 0) |
        (test_df["cpt_labels"].map(len) > 0)
    ].copy()

    if len(train_df) == 0:
        raise ValueError("No rows left in train split after label filtering.")

    if len(val_df) == 0:
        raise ValueError("No rows left in val split after label filtering.")

    if len(test_df) == 0:
        raise ValueError("No rows left in test split after label filtering.")

    icd_mlb = MultiLabelBinarizer(classes=sorted(kept_icd))
    cpt_mlb = MultiLabelBinarizer(classes=sorted(kept_cpt))

    train_icd_y = icd_mlb.fit_transform(train_df["icd_labels"])
    val_icd_y = icd_mlb.transform(val_df["icd_labels"])
    test_icd_y = icd_mlb.transform(test_df["icd_labels"])

    train_cpt_y = cpt_mlb.fit_transform(train_df["cpt_labels"])
    val_cpt_y = cpt_mlb.transform(val_df["cpt_labels"])
    test_cpt_y = cpt_mlb.transform(test_df["cpt_labels"])

    train_df["icd_vector"] = list(train_icd_y)
    val_df["icd_vector"] = list(val_icd_y)
    test_df["icd_vector"] = list(test_icd_y)

    train_df["cpt_vector"] = list(train_cpt_y)
    val_df["cpt_vector"] = list(val_cpt_y)
    test_df["cpt_vector"] = list(test_cpt_y)

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        icd_mlb,
        cpt_mlb
    )


# ============================================================
# DATASET CLASS WITH DYNAMIC PADDING
# ============================================================
class JointDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts = df["text"].tolist()
        self.icd_labels = df["icd_vector"].tolist()
        self.cpt_labels = df["cpt_vector"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "icd_labels": torch.tensor(self.icd_labels[idx], dtype=torch.float),
            "cpt_labels": torch.tensor(self.cpt_labels[idx], dtype=torch.float)
        }


class JointCollator:
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        texts = [item["text"] for item in batch]

        enc = self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        icd_labels = torch.stack([item["icd_labels"] for item in batch])
        cpt_labels = torch.stack([item["cpt_labels"] for item in batch])

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "icd_labels": icd_labels,
            "cpt_labels": cpt_labels
        }


# ============================================================
# MODEL
# ============================================================
class JointClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str,
        num_icd_labels: int,
        num_cpt_labels: int,
        dropout: float = 0.25
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.shared_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.icd_head = nn.Linear(hidden_size, num_icd_labels)
        self.cpt_head = nn.Linear(hidden_size, num_cpt_labels)

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        masked_hidden = last_hidden_state * mask

        summed = masked_hidden.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)

        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden_state = outputs.last_hidden_state

        cls_emb = last_hidden_state[:, 0, :]
        mean_emb = self.mean_pool(last_hidden_state, attention_mask)

        # CLS + mean pooling
        x = 0.5 * cls_emb + 0.5 * mean_emb

        x = self.dropout(x)
        x = self.shared_layer(x)

        icd_logits = self.icd_head(x)
        cpt_logits = self.cpt_head(x)

        return icd_logits, cpt_logits


# ============================================================
# LOSS HELPERS
# ============================================================
def compute_pos_weight(y: np.ndarray, clip_value: float = 1.5):
    positives = y.sum(axis=0)
    negatives = y.shape[0] - positives

    pos_weight = negatives / np.maximum(positives, 1.0)
    pos_weight = np.clip(pos_weight, 1.0, clip_value)

    return torch.tensor(pos_weight, dtype=torch.float)


def build_criteria(train_df, device):
    train_icd_y = np.vstack(train_df["icd_vector"].values)
    train_cpt_y = np.vstack(train_df["cpt_vector"].values)

    if CONFIG["use_icd_pos_weight"]:
        icd_pos_weight = compute_pos_weight(
            train_icd_y,
            CONFIG["icd_pos_weight_clip"]
        ).to(device)

        icd_criterion = nn.BCEWithLogitsLoss(pos_weight=icd_pos_weight)
    else:
        icd_criterion = nn.BCEWithLogitsLoss()

    if CONFIG["use_cpt_pos_weight"]:
        cpt_pos_weight = compute_pos_weight(
            train_cpt_y,
            CONFIG["cpt_pos_weight_clip"]
        ).to(device)

        cpt_criterion = nn.BCEWithLogitsLoss(pos_weight=cpt_pos_weight)
    else:
        cpt_criterion = nn.BCEWithLogitsLoss()

    return icd_criterion, cpt_criterion


# ============================================================
# METRICS AND DECODING
# ============================================================
def precision_at_k(y_true, y_probs, k=5):
    if y_probs.shape[1] == 0:
        return 0.0

    k = min(k, y_probs.shape[1])
    topk = np.argsort(-y_probs, axis=1)[:, :k]

    scores = []

    for i in range(y_true.shape[0]):
        true_set = set(np.where(y_true[i] == 1)[0].tolist())
        pred_set = set(topk[i].tolist())
        scores.append(len(true_set & pred_set) / k)

    return float(np.mean(scores))


def decode_predictions(
    y_probs,
    threshold,
    top_k,
    max_predictions,
    min_predictions
):
    n_samples, n_labels = y_probs.shape
    y_pred = np.zeros_like(y_probs, dtype=int)

    top_k = min(top_k, n_labels)
    max_predictions = min(max_predictions, n_labels)
    min_predictions = min(min_predictions, n_labels)

    for i in range(n_samples):
        probs = y_probs[i]

        top_indices = np.argsort(-probs)[:top_k].tolist()
        above_threshold = np.where(probs >= threshold)[0].tolist()

        top_set = set(top_indices)

        candidate_indices = [
            idx for idx in above_threshold
            if idx in top_set
        ]

        if len(candidate_indices) > max_predictions:
            candidate_indices = sorted(
                candidate_indices,
                key=lambda idx: probs[idx],
                reverse=True
            )[:max_predictions]

        if min_predictions > 0 and len(candidate_indices) < min_predictions:
            forced = np.argsort(-probs)[:min_predictions].tolist()
            candidate_indices = list(set(candidate_indices + forced))

        y_pred[i, candidate_indices] = 1

    return y_pred


def compute_metrics_from_predictions(y_true, y_pred, y_probs, prefix=""):
    return {
        f"{prefix}micro_f1": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),
        f"{prefix}macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        f"{prefix}micro_precision": precision_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),
        f"{prefix}micro_recall": recall_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),
        f"{prefix}precision_at_5": precision_at_k(y_true, y_probs, 5),
        f"{prefix}precision_at_8": precision_at_k(y_true, y_probs, 8),
    }


def tune_decode_params(
    y_true,
    y_probs,
    threshold_grid,
    topk_grid,
    max_predictions_grid,
    min_predictions,
    min_precision,
    task_name
):
    best_valid_score = -1.0
    best_valid_params = None
    best_valid_pred = None

    best_any_score = -1.0
    best_any_params = None
    best_any_pred = None

    rows = []

    for threshold in threshold_grid:
        for top_k in topk_grid:
            for max_predictions in max_predictions_grid:
                y_pred = decode_predictions(
                    y_probs=y_probs,
                    threshold=threshold,
                    top_k=top_k,
                    max_predictions=max_predictions,
                    min_predictions=min_predictions
                )

                micro_f1 = f1_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )

                micro_precision = precision_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )

                micro_recall = recall_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )

                row = {
                    "task": task_name,
                    "threshold": float(threshold),
                    "top_k": int(top_k),
                    "max_predictions": int(max_predictions),
                    "min_predictions": int(min_predictions),
                    "micro_f1": float(micro_f1),
                    "micro_precision": float(micro_precision),
                    "micro_recall": float(micro_recall)
                }

                rows.append(row)

                if micro_f1 > best_any_score:
                    best_any_score = micro_f1
                    best_any_params = row
                    best_any_pred = y_pred

                if micro_precision >= min_precision and micro_f1 > best_valid_score:
                    best_valid_score = micro_f1
                    best_valid_params = row
                    best_valid_pred = y_pred

    tuning_df = pd.DataFrame(rows)

    if best_valid_params is not None:
        return best_valid_params, best_valid_pred, tuning_df

    return best_any_params, best_any_pred, tuning_df


@torch.no_grad()
def collect_predictions(model, loader, device, icd_criterion=None, cpt_criterion=None):
    model.eval()

    all_icd_probs = []
    all_icd_labels = []
    all_cpt_probs = []
    all_cpt_labels = []

    total_loss = 0.0
    total_icd_loss = 0.0
    total_cpt_loss = 0.0

    use_loss = icd_criterion is not None and cpt_criterion is not None

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        icd_labels = batch["icd_labels"].to(device)
        cpt_labels = batch["cpt_labels"].to(device)

        icd_logits, cpt_logits = model(input_ids, attention_mask)

        if use_loss:
            icd_loss = icd_criterion(icd_logits, icd_labels)
            cpt_loss = cpt_criterion(cpt_logits, cpt_labels)

            loss = (
                CONFIG["icd_loss_weight"] * icd_loss +
                CONFIG["cpt_loss_weight"] * cpt_loss
            )

            total_loss += loss.item()
            total_icd_loss += icd_loss.item()
            total_cpt_loss += cpt_loss.item()

        icd_probs = torch.sigmoid(icd_logits)
        cpt_probs = torch.sigmoid(cpt_logits)

        all_icd_probs.append(icd_probs.detach().cpu().numpy())
        all_icd_labels.append(icd_labels.detach().cpu().numpy())

        all_cpt_probs.append(cpt_probs.detach().cpu().numpy())
        all_cpt_labels.append(cpt_labels.detach().cpu().numpy())

    y_icd_probs = np.vstack(all_icd_probs)
    y_icd_true = np.vstack(all_icd_labels)

    y_cpt_probs = np.vstack(all_cpt_probs)
    y_cpt_true = np.vstack(all_cpt_labels)

    losses = {
        "loss": total_loss / max(1, len(loader)) if use_loss else 0.0,
        "icd_loss": total_icd_loss / max(1, len(loader)) if use_loss else 0.0,
        "cpt_loss": total_cpt_loss / max(1, len(loader)) if use_loss else 0.0
    }

    return y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, losses


def evaluate(model, loader, device, icd_criterion=None, cpt_criterion=None, tune_thresholds=False):
    y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, losses = collect_predictions(
        model=model,
        loader=loader,
        device=device,
        icd_criterion=icd_criterion,
        cpt_criterion=cpt_criterion
    )

    if tune_thresholds:
        best_icd_params, y_icd_pred, icd_tuning_df = tune_decode_params(
            y_true=y_icd_true,
            y_probs=y_icd_probs,
            threshold_grid=CONFIG["icd_threshold_grid"],
            topk_grid=CONFIG["icd_topk_grid"],
            max_predictions_grid=CONFIG["icd_max_predictions_grid"],
            min_predictions=CONFIG["icd_min_predictions"],
            min_precision=CONFIG["icd_min_val_precision"],
            task_name="icd"
        )

        best_cpt_params, y_cpt_pred, cpt_tuning_df = tune_decode_params(
            y_true=y_cpt_true,
            y_probs=y_cpt_probs,
            threshold_grid=CONFIG["cpt_threshold_grid"],
            topk_grid=CONFIG["cpt_topk_grid"],
            max_predictions_grid=CONFIG["cpt_max_predictions_grid"],
            min_predictions=CONFIG["cpt_min_predictions"],
            min_precision=CONFIG["cpt_min_val_precision"],
            task_name="cpt"
        )

    else:
        best_icd_params = {
            "task": "icd",
            "threshold": CONFIG["icd_threshold"],
            "top_k": 10,
            "max_predictions": 10,
            "min_predictions": 1
        }

        best_cpt_params = {
            "task": "cpt",
            "threshold": CONFIG["cpt_threshold"],
            "top_k": 5,
            "max_predictions": 5,
            "min_predictions": 0
        }

        y_icd_pred = decode_predictions(
            y_icd_probs,
            threshold=CONFIG["icd_threshold"],
            top_k=10,
            max_predictions=10,
            min_predictions=1
        )

        y_cpt_pred = decode_predictions(
            y_cpt_probs,
            threshold=CONFIG["cpt_threshold"],
            top_k=5,
            max_predictions=5,
            min_predictions=0
        )

        icd_tuning_df = pd.DataFrame()
        cpt_tuning_df = pd.DataFrame()

    metrics = {}

    metrics.update(
        compute_metrics_from_predictions(
            y_true=y_icd_true,
            y_pred=y_icd_pred,
            y_probs=y_icd_probs,
            prefix="icd_"
        )
    )

    metrics.update(
        compute_metrics_from_predictions(
            y_true=y_cpt_true,
            y_pred=y_cpt_pred,
            y_probs=y_cpt_probs,
            prefix="cpt_"
        )
    )

    metrics.update(losses)

    metrics["joint_score"] = (
        CONFIG["selection_icd_weight"] * metrics["icd_micro_f1"] +
        CONFIG["selection_cpt_weight"] * metrics["cpt_micro_f1"]
    )

    return (
        metrics,
        y_icd_true,
        y_icd_probs,
        y_cpt_true,
        y_cpt_probs,
        best_icd_params,
        best_cpt_params,
        icd_tuning_df,
        cpt_tuning_df
    )


def apply_decode_params(y_probs, params):
    return decode_predictions(
        y_probs=y_probs,
        threshold=float(params["threshold"]),
        top_k=int(params["top_k"]),
        max_predictions=int(params["max_predictions"]),
        min_predictions=int(params["min_predictions"])
    )


# ============================================================
# TRAIN
# ============================================================
def train_model(model, train_loader, val_loader, train_df):
    device = CONFIG["device"]
    model = model.to(device)

    icd_criterion, cpt_criterion = build_criteria(train_df, device)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.encoder.parameters(),
                "lr": CONFIG["encoder_lr"]
            },
            {
                "params": (
                    list(model.shared_layer.parameters()) +
                    list(model.icd_head.parameters()) +
                    list(model.cpt_head.parameters())
                ),
                "lr": CONFIG["classifier_lr"]
            },
        ],
        weight_decay=CONFIG["weight_decay"]
    )

    update_steps_per_epoch = int(np.ceil(len(train_loader) / CONFIG["grad_accum_steps"]))
    total_steps = update_steps_per_epoch * CONFIG["epochs"]
    warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    use_amp = CONFIG["fp16"] and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_selection_score = -1.0
    best_state_dict = None
    best_icd_params = None
    best_cpt_params = None
    best_icd_tuning_df = None
    best_cpt_tuning_df = None

    patience_counter = 0
    history = []

    for epoch in range(CONFIG["epochs"]):
        model.train()
        total_train_loss = 0.0

        optimizer.zero_grad()

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{CONFIG['epochs']}"
        )

        for step, batch in enumerate(progress, start=1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            icd_labels = batch["icd_labels"].to(device)
            cpt_labels = batch["cpt_labels"].to(device)

            with torch.cuda.amp.autocast(enabled=use_amp):
                icd_logits, cpt_logits = model(input_ids, attention_mask)

                icd_loss = icd_criterion(icd_logits, icd_labels)
                cpt_loss = cpt_criterion(cpt_logits, cpt_labels)

                loss = (
                    CONFIG["icd_loss_weight"] * icd_loss +
                    CONFIG["cpt_loss_weight"] * cpt_loss
                )

                loss = loss / CONFIG["grad_accum_steps"]

            scaler.scale(loss).backward()

            if step % CONFIG["grad_accum_steps"] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    CONFIG["grad_clip"]
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                optimizer.zero_grad()

            total_train_loss += loss.item() * CONFIG["grad_accum_steps"]
            avg_loss = total_train_loss / step

            progress.set_postfix({"train_loss": f"{avg_loss:.4f}"})

        train_loss = total_train_loss / max(1, len(train_loader))

        (
            val_metrics,
            _,
            _,
            _,
            _,
            val_icd_params,
            val_cpt_params,
            icd_tuning_df,
            cpt_tuning_df
        ) = evaluate(
            model=model,
            loader=val_loader,
            device=device,
            icd_criterion=icd_criterion,
            cpt_criterion=cpt_criterion,
            tune_thresholds=True
        )

        print("\n" + "=" * 80)
        print(f"Epoch {epoch + 1}")
        print(f"Train Loss      : {train_loss:.4f}")
        print(f"Val Total Loss  : {val_metrics['loss']:.4f}")
        print(f"Val ICD Loss    : {val_metrics['icd_loss']:.4f}")
        print(f"Val CPT Loss    : {val_metrics['cpt_loss']:.4f}")
        print(f"Val ICD MicroF1 : {val_metrics['icd_micro_f1']:.4f}")
        print(f"Val ICD Prec    : {val_metrics['icd_micro_precision']:.4f}")
        print(f"Val ICD Recall  : {val_metrics['icd_micro_recall']:.4f}")
        print(f"Val CPT MicroF1 : {val_metrics['cpt_micro_f1']:.4f}")
        print(f"Val CPT Prec    : {val_metrics['cpt_micro_precision']:.4f}")
        print(f"Val CPT Recall  : {val_metrics['cpt_micro_recall']:.4f}")
        print(f"Val Joint Score : {val_metrics['joint_score']:.4f}")
        print(f"Best ICD Params : {val_icd_params}")
        print(f"Best CPT Params : {val_cpt_params}")
        print("=" * 80)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()}
        })

        # ICD-focused checkpoint selection.
        # Main objective: maximize ICD F1.
        # Constraint: CPT should not collapse below the requirement.
        icd_f1 = val_metrics["icd_micro_f1"]
        cpt_f1 = val_metrics["cpt_micro_f1"]

        cpt_penalty = 0.0
        if cpt_f1 < 0.4643:
            cpt_penalty = 0.4643 - cpt_f1

        icd_precision_penalty = 0.0
        if val_metrics["icd_micro_precision"] < CONFIG["icd_min_val_precision"]:
            icd_precision_penalty = 0.25 * (
                CONFIG["icd_min_val_precision"] -
                val_metrics["icd_micro_precision"]
            )

        # This is intentionally ICD-first.
        selection_score = icd_f1 - cpt_penalty - icd_precision_penalty

        if selection_score > best_selection_score:
            best_selection_score = selection_score
            best_state_dict = copy.deepcopy(model.state_dict())
            best_icd_params = copy.deepcopy(val_icd_params)
            best_cpt_params = copy.deepcopy(val_cpt_params)
            best_icd_tuning_df = icd_tuning_df.copy()
            best_cpt_tuning_df = cpt_tuning_df.copy()
            patience_counter = 0

            print("Saved new best joint model in memory.")

        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{CONFIG['patience']}")

            if patience_counter >= CONFIG["patience"]:
                print("Early stopping triggered.")
                break

    if best_state_dict is None:
        raise RuntimeError("Training finished but no best model state was captured.")

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        os.path.join(CONFIG["output_dir"], "training_history.csv"),
        index=False
    )

    if best_icd_tuning_df is not None:
        best_icd_tuning_df.to_csv(
            os.path.join(CONFIG["output_dir"], "icd_validation_tuning_results.csv"),
            index=False
        )

    if best_cpt_tuning_df is not None:
        best_cpt_tuning_df.to_csv(
            os.path.join(CONFIG["output_dir"], "cpt_validation_tuning_results.csv"),
            index=False
        )

    return (
        best_state_dict,
        history_df,
        best_icd_params,
        best_cpt_params,
        icd_criterion,
        cpt_criterion
    )


# ============================================================
# FINAL TEST EVALUATION USING VALIDATION SELECTED PARAMS
# ============================================================
def evaluate_test_with_best_params(
    model,
    test_loader,
    device,
    icd_criterion,
    cpt_criterion,
    best_icd_params,
    best_cpt_params
):
    y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, losses = collect_predictions(
        model=model,
        loader=test_loader,
        device=device,
        icd_criterion=icd_criterion,
        cpt_criterion=cpt_criterion
    )

    y_icd_pred = apply_decode_params(y_icd_probs, best_icd_params)
    y_cpt_pred = apply_decode_params(y_cpt_probs, best_cpt_params)

    metrics = {}

    metrics.update(
        compute_metrics_from_predictions(
            y_true=y_icd_true,
            y_pred=y_icd_pred,
            y_probs=y_icd_probs,
            prefix="icd_"
        )
    )

    metrics.update(
        compute_metrics_from_predictions(
            y_true=y_cpt_true,
            y_pred=y_cpt_pred,
            y_probs=y_cpt_probs,
            prefix="cpt_"
        )
    )

    metrics.update(losses)

    metrics["joint_score"] = (
        CONFIG["selection_icd_weight"] * metrics["icd_micro_f1"] +
        CONFIG["selection_cpt_weight"] * metrics["cpt_micro_f1"]
    )

    return metrics, y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs


# ============================================================
# MAIN
# ============================================================
def main():
    print(f"Using device: {CONFIG['device']}")

    train_raw = load_split_from_folder(CONFIG["train_dir"], "train")
    val_raw = load_split_from_folder(CONFIG["val_dir"], "val")
    test_raw = load_split_from_folder(CONFIG["test_dir"], "test")

    train_df, val_df, test_df, icd_mlb, cpt_mlb = build_datasets(
        train_raw,
        val_raw,
        test_raw
    )

    print("\nAfter preprocessing and label filtering:")
    print(f"Train shape: {train_df.shape}")
    print(f"Val shape  : {val_df.shape}")
    print(f"Test shape : {test_df.shape}")
    print(f"Number of ICD labels: {len(icd_mlb.classes_)}")
    print(f"Number of CPT labels: {len(cpt_mlb.classes_)}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["pretrained_model"])

    train_dataset = JointDataset(train_df)
    val_dataset = JointDataset(val_df)
    test_dataset = JointDataset(test_df)

    collator = JointCollator(
        tokenizer=tokenizer,
        max_length=CONFIG["max_length"]
    )

    pin_memory = CONFIG["device"] == "cuda"

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        collate_fn=collator,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        collate_fn=collator,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        collate_fn=collator,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory
    )

    model = JointClassifier(
        pretrained_model_name=CONFIG["pretrained_model"],
        num_icd_labels=len(icd_mlb.classes_),
        num_cpt_labels=len(cpt_mlb.classes_),
        dropout=CONFIG["dropout"]
    )

    print("\nTraining joint ICD + CPT model...")

    (
        best_state_dict,
        history_df,
        best_icd_params,
        best_cpt_params,
        icd_criterion,
        cpt_criterion
    ) = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        train_df=train_df
    )

    print("\nLoading best joint model from memory...")
    model.load_state_dict(best_state_dict)
    model.to(CONFIG["device"])

    print("Evaluating joint model on test set using validation selected thresholds...")

    test_metrics, y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs = evaluate_test_with_best_params(
        model=model,
        test_loader=test_loader,
        device=CONFIG["device"],
        icd_criterion=icd_criterion,
        cpt_criterion=cpt_criterion,
        best_icd_params=best_icd_params,
        best_cpt_params=best_cpt_params
    )

    print("\n" + "=" * 80)
    print("JOINT ICD + CPT TEST RESULTS")
    print("=" * 80)

    for k, v in test_metrics.items():
        print(f"{k:22s}: {v:.4f}")

    print("=" * 80)

    print("\nTarget check:")
    print(
        f"ICD Micro F1 > 0.4342 : "
        f"{'PASSED' if test_metrics['icd_micro_f1'] > 0.4342 else 'NOT PASSED'}"
    )

    print(
        f"CPT Micro F1 > 0.4643 : "
        f"{'PASSED' if test_metrics['cpt_micro_f1'] > 0.4643 else 'NOT PASSED'}"
    )

    output_dir = CONFIG["output_dir"]

    # Keep same original file names
    with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=2)

    with open(os.path.join(output_dir, "icd_label_classes.json"), "w", encoding="utf-8") as f:
        json.dump(icd_mlb.classes_.tolist(), f, indent=2)

    with open(os.path.join(output_dir, "cpt_label_classes.json"), "w", encoding="utf-8") as f:
        json.dump(cpt_mlb.classes_.tolist(), f, indent=2)

    with open(os.path.join(output_dir, "test_metrics.json"), "w", encoding="utf-8") as f:
        json.dump({k: float(v) for k, v in test_metrics.items()}, f, indent=2)

    np.save(os.path.join(output_dir, "test_y_icd_true.npy"), y_icd_true)
    np.save(os.path.join(output_dir, "test_y_icd_probs.npy"), y_icd_probs)

    np.save(os.path.join(output_dir, "test_y_cpt_true.npy"), y_cpt_true)
    np.save(os.path.join(output_dir, "test_y_cpt_probs.npy"), y_cpt_probs)

    train_df.to_csv(os.path.join(output_dir, "train_data.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir, "val_data.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, "test_data.csv"), index=False)

    history_df.to_csv(
        os.path.join(output_dir, "training_history.csv"),
        index=False
    )

    safe_state_dict = make_state_dict_safe_for_saving(best_state_dict)

    save_file(
        safe_state_dict,
        os.path.join(output_dir, "best_model.safetensors")
    )

    # Extra helpful files while preserving original names
    with open(os.path.join(output_dir, "best_icd_decode_params.json"), "w", encoding="utf-8") as f:
        json.dump(best_icd_params, f, indent=2)

    with open(os.path.join(output_dir, "best_cpt_decode_params.json"), "w", encoding="utf-8") as f:
        json.dump(best_cpt_params, f, indent=2)

    final_metrics_table = pd.DataFrame([
        {
            "Model": "Model 1 BioClinicalBERT Joint ICD + CPT ICD Focused",
            "ICD Micro F1": test_metrics["icd_micro_f1"],
            "ICD Macro F1": test_metrics["icd_macro_f1"],
            "ICD Micro Precision": test_metrics["icd_micro_precision"],
            "ICD Micro Recall": test_metrics["icd_micro_recall"],
            "ICD P@5": test_metrics["icd_precision_at_5"],
            "ICD P@8": test_metrics["icd_precision_at_8"],
            "CPT Micro F1": test_metrics["cpt_micro_f1"],
            "CPT Macro F1": test_metrics["cpt_macro_f1"],
            "CPT Micro Precision": test_metrics["cpt_micro_precision"],
            "CPT Micro Recall": test_metrics["cpt_micro_recall"],
            "CPT P@5": test_metrics["cpt_precision_at_5"],
            "CPT P@8": test_metrics["cpt_precision_at_8"],
            "Joint Score": test_metrics["joint_score"]
        }
    ])

    final_metrics_table.to_csv(
        os.path.join(output_dir, "final_metrics_table.csv"),
        index=False
    )

    print(f"\nAll outputs saved successfully to: {output_dir}")
    print(f"Training history rows: {len(history_df)}")
    print(f"Best ICD decode params: {best_icd_params}")
    print(f"Best CPT decode params: {best_cpt_params}")


if __name__ == "__main__":
    main()

c:\Users\samsa\Documents\Testing NLP Temporal Coding\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All outputs will be saved to: ./model1_joint_icd_cpt_output
Using device: cuda

Loading train files from: data/train
 - train1.jsonl
 - train2.jsonl
 - train3.jsonl
 - train4.jsonl
Loaded train1.jsonl -> shape: (10000, 8)
Loaded train2.jsonl -> shape: (10000, 8)
Loaded train3.jsonl -> shape: (10000, 8)
Loaded train4.jsonl -> shape: (7617, 8)
Combined train shape: (37617, 8)

Loading val files from: data/val
 - val.jsonl
Loaded val.jsonl -> shape: (4715, 8)
Combined val shape: (4715, 8)

Loading test files from: data/test
 - test.jsonl
Loaded test.jsonl -> shape: (4628, 8)
Combined test shape: (4628, 8)

After preprocessing and label filtering:
Train shape: (36950, 13)
Val shape  : (4633, 13)
Test shape : (4534, 13)
Number of ICD labels: 50
Number of CPT labels: 50


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29037.94it/s]
BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Training joint ICD + CPT model...


Epoch 1/7: 100%|██████████| 4619/4619 [08:30<00:00,  9.05it/s, train_loss=0.8018]


...

Epoch 5/7: 100%|██████████| 4619/4619 [07:53<00:00,  9.76it/s, train_loss=0.6051]



Epoch 5
Train Loss      : 0.6051
Val Total Loss  : 0.6622
Val ICD Loss    : 0.3769
Val CPT Loss    : 0.1704
Val ICD MicroF1 : 0.4423
Val ICD Prec    : 0.4172
Val ICD Recall  : 0.4706
Val CPT MicroF1 : 0.5423
Val CPT Prec    : 0.5903
Val CPT Recall  : 0.5016
Val Joint Score : 0.4523
Best ICD Params : {'task': 'icd', 'threshold': 0.32, 'top_k': 15, 'max_predictions': 15, 'min_predictions': 1, 'micro_f1': 0.4422507747484244, 'micro_precision': 0.4171511150523861, 'micro_recall': 0.47056426216146124}
Best CPT Params : {'task': 'cpt', 'threshold': 0.32, 'top_k': 6, 'max_predictions': 6, 'min_predictions': 0, 'micro_f1': 0.5423162583518931, 'micro_precision': 0.5902672565299073, 'micro_recall': 0.5015706267058035}
No improvement. Patience: 1/2


Epoch 6/7: 100%|██████████| 4619/4619 [07:50<00:00,  9.81it/s, train_loss=0.5885]



Epoch 6
Train Loss      : 0.5885
Val Total Loss  : 0.6784
Val ICD Loss    : 0.3888
Val CPT Loss    : 0.1706
Val ICD MicroF1 : 0.4442
Val ICD Prec    : 0.4098
Val ICD Recall  : 0.4848
Val CPT MicroF1 : 0.5417
Val CPT Prec    : 0.6005
Val CPT Recall  : 0.4934
Val Joint Score : 0.4539
Best ICD Params : {'task': 'icd', 'threshold': 0.3, 'top_k': 15, 'max_predictions': 15, 'min_predictions': 1, 'micro_f1': 0.44419551934826884, 'micro_precision': 0.4098468476933195, 'micro_recall': 0.48482827609203066}
Best CPT Params : {'task': 'cpt', 'threshold': 0.32, 'top_k': 6, 'max_predictions': 6, 'min_predictions': 0, 'micro_f1': 0.5417078893066116, 'micro_precision': 0.6004511843589422, 'micro_recall': 0.49343426541016533}
No improvement. Patience: 2/2
Early stopping triggered.

Loading best joint model from memory...
Evaluating joint model on test set using validation selected thresholds...



JOINT ICD + CPT TEST RESULTS
icd_micro_f1          : 0.4506
icd_macro_f1          : 0.3130
icd_micro_precision   : 0.4006
icd_micro_recall      : 0.5148
icd_precision_at_5    : 0.4315
icd_precision_at_8    : 0.3621
cpt_micro_f1          : 0.5380
cpt_macro_f1          : 0.1751
cpt_micro_precision   : 0.5903
cpt_micro_recall      : 0.4942
cpt_precision_at_5    : 0.4775
cpt_precision_at_8    : 0.3799
loss                  : 0.6547
icd_loss              : 0.3690
cpt_loss              : 0.1739
joint_score           : 0.4593

Target check:
ICD Micro F1 > 0.4342 : PASSED
CPT Micro F1 > 0.4643 : PASSED

All outputs saved successfully to: ./model1_joint_icd_cpt_output
Training history rows: 6
Best ICD decode params: {'task': 'icd', 'threshold': 0.3, 'top_k': 15, 'max_predictions': 15, 'min_predictions': 1, 'micro_f1': 0.4455827546011263, 'micro_precision': 0.40008255203726634, 'micro_recall': 0.5027601793190323}
Best CPT decode params: {'task': 'cpt', 'threshold': 0.32, 'top_k': 6, 'max_predic

### Model 2 ICD + CPT 

In [2]:
# ============================================================
# MODEL 2: JOINT ICD + CPT CODE PREDICTION USING LONGFORMER
# FOLDER-BASED VERSION
#
# Uses:
#   - Training data from:   data/train/train1.jsonl, train2.jsonl, ...
#   - Validation data from: data/val/val.jsonl
#   - Test data from:       data/test/test.jsonl
#
# Expected JSONL row format:
#   {
#       "discharge_text": ...,
#       "discharge_narrative": ...,
#       "events": [...],
#       "labels": {
#           "icd10": [...],
#           "cpt": [...]
#       }
#   }
#
# Model 2 setup:
#   - discharge summary + radiology notes
#   - radiology sorted by relative_time_hrs
#   - merged into one long input
#   - Longformer for joint multi-label ICD + CPT prediction
#
# Output:
#   ./model2_joint_icd_cpt_output
# ============================================================

import os
import re
import json
import copy
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from safetensors.torch import save_file

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    "train_dir": "data/train",
    "val_dir": "data/val",
    "test_dir": "data/test",
    "output_dir": "./model2_joint_icd_cpt_output",
    "pretrained_model": "allenai/longformer-base-4096",
    "max_length": 1024,          # reduce to 512 if memory is tight
    "batch_size": 2,             # reduce to 1 if CUDA OOM
    "epochs": 3,
    "encoder_lr": 2e-5,
    "classifier_lr": 1e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout": 0.3,
    "grad_clip": 1.0,
    "icd_threshold": 0.3,
    "cpt_threshold": 0.3,
    "min_icd_label_freq": 1,     # use 1 for small/sample runs
    "min_cpt_label_freq": 1,     # use 1 for small/sample runs
    "top_k_icd_labels": 50,      # set None to keep all ICD labels
    "top_k_cpt_labels": 50,      # set None to keep all CPT labels
    "num_workers": 0,            # safer on Windows
    "fp16": True,
    "seed": 42,
    "max_radiology_notes": 3,    # reduce to 2 for speed
    "icd_loss_weight": 1.0,
    "cpt_loss_weight": 1.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("All outputs will be saved to:", CONFIG["output_dir"])
print("Using device:", CONFIG["device"])


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])


# ============================================================
# HELPERS
# ============================================================
def natural_sort_key(path_obj: Path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path_obj.name)
    ]


def safe_str(x) -> str:
    return "" if x is None else str(x)


def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


# ============================================================
# FILE LOADING
# ============================================================
def get_jsonl_files(folder_path: str):
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    files = sorted(folder.glob("*.jsonl"), key=natural_sort_key)
    if len(files) == 0:
        raise FileNotFoundError(f"No .jsonl files found in: {folder_path}")

    return files


def load_jsonl_file(file_path: Path) -> pd.DataFrame:
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in {file_path} at line {line_num}: {e}") from e

    if len(rows) == 0:
        raise ValueError(f"No rows found in file: {file_path}")

    return pd.DataFrame(rows)


def load_split_from_folder(folder_path: str, split_name: str) -> pd.DataFrame:
    files = get_jsonl_files(folder_path)

    print(f"\nLoading {split_name} files from: {folder_path}")
    for f in files:
        print(" -", f.name)

    dfs = []
    for file_path in files:
        df = load_jsonl_file(file_path)
        print(f"Loaded {file_path.name} -> shape: {df.shape}")
        dfs.append(df)

    split_df = pd.concat(dfs, ignore_index=True)
    print(f"Combined {split_name} shape: {split_df.shape}")
    return split_df


# ============================================================
# TEXT BUILDING
# ============================================================
def build_model2_text(row) -> str:
    """
    Build Model 2 input:
    - discharge summary
    - radiology notes sorted by relative_time_hrs
    """
    sections = []

    discharge = safe_str(
        row.get("discharge_text") or row.get("discharge_narrative", "")
    ).strip()

    if discharge:
        sections.append("DISCHARGE SUMMARY:\n" + " ".join(discharge.split()))

    events = row.get("events") or []
    if isinstance(events, list):
        radiology_notes = []

        for e in events:
            event_type = safe_str(e.get("event_type", "")).lower().strip()
            if event_type != "radiology":
                continue

            text = safe_str(e.get("text", "")).strip()
            if not text:
                continue

            t = safe_float(e.get("relative_time_hrs"))
            if t is None:
                t = float("inf")

            radiology_notes.append({
                "t": t,
                "text": " ".join(text.split())
            })

        radiology_notes.sort(key=lambda x: x["t"])
        radiology_notes = radiology_notes[:CONFIG["max_radiology_notes"]]

        if radiology_notes:
            parts = []
            for i, note in enumerate(radiology_notes, start=1):
                if np.isfinite(note["t"]):
                    header = f"RADIOLOGY NOTE {i} AT {note['t']:.2f} HOURS:"
                else:
                    header = f"RADIOLOGY NOTE {i}:"
                parts.append(f"{header}\n{note['text']}")
            sections.append("\n\n".join(parts))

    return "\n\n".join([s for s in sections if s]).strip()


# ============================================================
# LABEL HANDLING
# ============================================================
def _item_to_str(item) -> str:
    if isinstance(item, dict):
        for key in ("code", "label", "value", "name"):
            val = safe_str(item.get(key, "")).strip()
            if val:
                return val
        return ""
    return safe_str(item).strip()


def normalize_label_list(x) -> list:
    if x is None:
        return []

    if isinstance(x, list):
        return [s for item in x for s in [_item_to_str(item)] if s]

    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        for sep in (",", ";", "|"):
            if sep in x:
                return [p.strip() for p in x.split(sep) if p.strip()]
        return [x]

    val = safe_str(x).strip()
    return [val] if val else []


def extract_icd_labels(label_obj) -> list:
    if not isinstance(label_obj, dict):
        return []
    return normalize_label_list(label_obj.get("icd10", []))


def extract_cpt_labels(label_obj) -> list:
    if not isinstance(label_obj, dict):
        return []
    return normalize_label_list(label_obj.get("cpt", []))


def get_kept_labels(df_train: pd.DataFrame, label_col: str, min_freq: int, top_k):
    counter = Counter()
    for labels in df_train[label_col]:
        counter.update(labels)

    kept = sorted(
        (lab for lab, freq in counter.items() if freq >= min_freq),
        key=lambda x: counter[x],
        reverse=True,
    )
    return kept[:top_k] if top_k is not None else kept


# ============================================================
# DATA PREP
# ============================================================
def preprocess_split(df_raw: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if "labels" not in df_raw.columns:
        raise ValueError(f"{split_name} dataset must contain a 'labels' column.")

    df = df_raw.copy()
    df["text"] = df.apply(build_model2_text, axis=1).fillna("").astype(str)
    df["icd_labels"] = df["labels"].apply(extract_icd_labels)
    df["cpt_labels"] = df["labels"].apply(extract_cpt_labels)

    df = df[df["text"].str.len() > 0]
    df = df[(df["icd_labels"].map(len) > 0) | (df["cpt_labels"].map(len) > 0)]

    if df.empty:
        raise ValueError(f"No usable rows left after preprocessing for {split_name}.")

    return df.reset_index(drop=True)


def build_datasets(train_raw: pd.DataFrame, val_raw: pd.DataFrame, test_raw: pd.DataFrame):
    train_df = preprocess_split(train_raw, "train")
    val_df = preprocess_split(val_raw, "val")
    test_df = preprocess_split(test_raw, "test")

    kept_icd = get_kept_labels(
        train_df,
        label_col="icd_labels",
        min_freq=CONFIG["min_icd_label_freq"],
        top_k=CONFIG["top_k_icd_labels"],
    )
    kept_cpt = get_kept_labels(
        train_df,
        label_col="cpt_labels",
        min_freq=CONFIG["min_cpt_label_freq"],
        top_k=CONFIG["top_k_cpt_labels"],
    )

    if not kept_icd:
        raise ValueError("No ICD labels left after filtering.")
    if not kept_cpt:
        raise ValueError("No CPT labels left after filtering.")

    kept_icd_set = set(kept_icd)
    kept_cpt_set = set(kept_cpt)

    for split in (train_df, val_df, test_df):
        split["icd_labels"] = split["icd_labels"].apply(lambda labs: [x for x in labs if x in kept_icd_set])
        split["cpt_labels"] = split["cpt_labels"].apply(lambda labs: [x for x in labs if x in kept_cpt_set])

    train_df = train_df[(train_df["icd_labels"].map(len) > 0) | (train_df["cpt_labels"].map(len) > 0)].copy()
    val_df = val_df[(val_df["icd_labels"].map(len) > 0) | (val_df["cpt_labels"].map(len) > 0)].copy()
    test_df = test_df[(test_df["icd_labels"].map(len) > 0) | (test_df["cpt_labels"].map(len) > 0)].copy()

    for split_name, split in (("train", train_df), ("val", val_df), ("test", test_df)):
        if split.empty:
            raise ValueError(f"No rows left in {split_name} split after label filtering.")

    icd_mlb = MultiLabelBinarizer(classes=sorted(kept_icd))
    cpt_mlb = MultiLabelBinarizer(classes=sorted(kept_cpt))

    train_icd_y = icd_mlb.fit_transform(train_df["icd_labels"])
    val_icd_y = icd_mlb.transform(val_df["icd_labels"])
    test_icd_y = icd_mlb.transform(test_df["icd_labels"])

    train_cpt_y = cpt_mlb.fit_transform(train_df["cpt_labels"])
    val_cpt_y = cpt_mlb.transform(val_df["cpt_labels"])
    test_cpt_y = cpt_mlb.transform(test_df["cpt_labels"])

    train_df["icd_vector"] = list(train_icd_y)
    val_df["icd_vector"] = list(val_icd_y)
    test_df["icd_vector"] = list(test_icd_y)

    train_df["cpt_vector"] = list(train_cpt_y)
    val_df["cpt_vector"] = list(val_cpt_y)
    test_df["cpt_vector"] = list(test_cpt_y)

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        icd_mlb,
        cpt_mlb,
    )


# ============================================================
# TOKENIZATION
# ============================================================
def batch_tokenize(texts: list, tokenizer, max_length: int):
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt",
    )


# ============================================================
# DATASET
# ============================================================
class Model2JointDataset(Dataset):
    def __init__(self, encodings, icd_labels: np.ndarray, cpt_labels: np.ndarray):
        self.encodings = encodings
        self.icd_labels = torch.from_numpy(icd_labels.astype(np.float32))
        self.cpt_labels = torch.from_numpy(cpt_labels.astype(np.float32))

    def __len__(self):
        return len(self.icd_labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "icd_labels": self.icd_labels[idx],
            "cpt_labels": self.cpt_labels[idx],
        }


# ============================================================
# MODEL
# ============================================================
class LongformerJointClassifier(nn.Module):
    def __init__(self, pretrained_model_name: str, num_icd_labels: int, num_cpt_labels: int, dropout: float = 0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.icd_head = nn.Linear(hidden_size, num_icd_labels)
        self.cpt_head = nn.Linear(hidden_size, num_cpt_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        icd_logits = self.icd_head(cls)
        cpt_logits = self.cpt_head(cls)
        return icd_logits, cpt_logits


# ============================================================
# METRICS
# ============================================================
def precision_at_k(y_true: np.ndarray, y_probs: np.ndarray, k: int = 5) -> float:
    y_true = y_true.astype(int)
    topk = np.argpartition(-y_probs, k, axis=1)[:, :k]
    hits = np.take_along_axis(y_true, topk, axis=1).sum(axis=1)
    return float(hits.mean()) / k


def compute_metrics(y_true: np.ndarray, y_probs: np.ndarray, threshold: float = 0.3, prefix: str = "") -> dict:
    y_true = y_true.astype(int)
    y_pred = (y_probs >= threshold).astype(int)

    return {
        f"{prefix}micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}precision_at_5": precision_at_k(y_true, y_probs, 5),
        f"{prefix}precision_at_8": precision_at_k(y_true, y_probs, 8),
    }


def find_best_threshold(y_true: np.ndarray, y_probs: np.ndarray):
    thresholds = np.arange(0.1, 0.9, 0.05)
    y_true_bool = y_true.astype(bool)
    preds = (y_probs[None] >= thresholds[:, None, None])

    tp = (preds & y_true_bool[None]).sum(axis=(1, 2))
    fp = (preds & ~y_true_bool[None]).sum(axis=(1, 2))
    fn = (~preds & y_true_bool[None]).sum(axis=(1, 2))

    precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1 = np.where(
        precision + recall > 0,
        2 * precision * recall / (precision + recall),
        0.0,
    )

    best_idx = int(f1.argmax())
    return float(thresholds[best_idx]), float(f1[best_idx])


# ============================================================
# EVALUATION
# ============================================================
@torch.no_grad()
def evaluate(model, loader, device, icd_threshold: float = 0.3, cpt_threshold: float = 0.3):
    model.eval()

    icd_criterion = nn.BCEWithLogitsLoss()
    cpt_criterion = nn.BCEWithLogitsLoss()

    all_icd_probs = []
    all_icd_labels = []
    all_cpt_probs = []
    all_cpt_labels = []

    total_loss = 0.0
    total_icd_loss = 0.0
    total_cpt_loss = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        icd_labels = batch["icd_labels"].to(device)
        cpt_labels = batch["cpt_labels"].to(device)

        icd_logits, cpt_logits = model(input_ids, attention_mask)

        icd_loss = icd_criterion(icd_logits, icd_labels)
        cpt_loss = cpt_criterion(cpt_logits, cpt_labels)
        loss = CONFIG["icd_loss_weight"] * icd_loss + CONFIG["cpt_loss_weight"] * cpt_loss

        total_loss += loss.item()
        total_icd_loss += icd_loss.item()
        total_cpt_loss += cpt_loss.item()

        all_icd_probs.append(torch.sigmoid(icd_logits).detach().cpu().numpy())
        all_icd_labels.append(icd_labels.detach().cpu().numpy())
        all_cpt_probs.append(torch.sigmoid(cpt_logits).detach().cpu().numpy())
        all_cpt_labels.append(cpt_labels.detach().cpu().numpy())

    y_icd_probs = np.vstack(all_icd_probs)
    y_icd_true = np.vstack(all_icd_labels)
    y_cpt_probs = np.vstack(all_cpt_probs)
    y_cpt_true = np.vstack(all_cpt_labels)

    metrics = {}
    metrics.update(compute_metrics(y_icd_true, y_icd_probs, icd_threshold, prefix="icd_"))
    metrics.update(compute_metrics(y_cpt_true, y_cpt_probs, cpt_threshold, prefix="cpt_"))
    metrics["loss"] = total_loss / max(1, len(loader))
    metrics["icd_loss"] = total_icd_loss / max(1, len(loader))
    metrics["cpt_loss"] = total_cpt_loss / max(1, len(loader))
    metrics["joint_score"] = (metrics["icd_micro_f1"] + metrics["cpt_micro_f1"]) / 2.0

    return metrics, y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs


# ============================================================
# TRAINING
# ============================================================
def train_model(model, train_loader, val_loader):
    device = CONFIG["device"]
    model = model.to(device)

    icd_criterion = nn.BCEWithLogitsLoss()
    cpt_criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": CONFIG["encoder_lr"]},
            {
                "params": list(model.icd_head.parameters()) + list(model.cpt_head.parameters()),
                "lr": CONFIG["classifier_lr"],
            },
        ],
        weight_decay=CONFIG["weight_decay"],
    )

    total_steps = len(train_loader) * CONFIG["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(CONFIG["warmup_ratio"] * total_steps),
        num_training_steps=total_steps,
    )

    use_amp = CONFIG["fp16"] and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_joint_score = -1.0
    best_state_dict = None
    best_icd_threshold = CONFIG["icd_threshold"]
    best_cpt_threshold = CONFIG["cpt_threshold"]
    history = []

    print("Starting training loop...")

    for epoch in range(CONFIG["epochs"]):
        model.train()
        total_train_loss = 0.0

        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{CONFIG['epochs']}")
        for step, batch in enumerate(progress, start=1):
            if step == 1:
                print("First batch loaded")

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            icd_labels = batch["icd_labels"].to(device)
            cpt_labels = batch["cpt_labels"].to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                icd_logits, cpt_logits = model(input_ids, attention_mask)
                icd_loss = icd_criterion(icd_logits, icd_labels)
                cpt_loss = cpt_criterion(cpt_logits, cpt_labels)
                loss = CONFIG["icd_loss_weight"] * icd_loss + CONFIG["cpt_loss_weight"] * cpt_loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_train_loss += loss.item()
            progress.set_postfix({"train_loss": f"{total_train_loss / step:.4f}"})

        train_loss = total_train_loss / max(1, len(train_loader))

        _, val_icd_true, val_icd_probs, val_cpt_true, val_cpt_probs = evaluate(
            model,
            val_loader,
            device,
            CONFIG["icd_threshold"],
            CONFIG["cpt_threshold"]
        )

        tuned_icd_threshold, _ = find_best_threshold(val_icd_true, val_icd_probs)
        tuned_cpt_threshold, _ = find_best_threshold(val_cpt_true, val_cpt_probs)

        val_metrics = {}
        val_metrics.update(compute_metrics(val_icd_true, val_icd_probs, tuned_icd_threshold, prefix="icd_"))
        val_metrics.update(compute_metrics(val_cpt_true, val_cpt_probs, tuned_cpt_threshold, prefix="cpt_"))
        val_metrics["joint_score"] = (val_metrics["icd_micro_f1"] + val_metrics["cpt_micro_f1"]) / 2.0

        print("\n" + "=" * 90)
        print(f"Epoch {epoch + 1}")
        print(f"Train Loss         : {train_loss:.4f}")
        print(f"Val ICD Threshold  : {tuned_icd_threshold:.2f}")
        print(f"Val CPT Threshold  : {tuned_cpt_threshold:.2f}")
        print(f"Val ICD Micro F1   : {val_metrics['icd_micro_f1']:.4f}")
        print(f"Val CPT Micro F1   : {val_metrics['cpt_micro_f1']:.4f}")
        print(f"Val Joint Score    : {val_metrics['joint_score']:.4f}")
        print("=" * 90)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_icd_threshold": tuned_icd_threshold,
            "val_cpt_threshold": tuned_cpt_threshold,
            **{f"val_{k}": v for k, v in val_metrics.items()}
        })

        if val_metrics["joint_score"] > best_joint_score:
            best_joint_score = val_metrics["joint_score"]
            best_icd_threshold = tuned_icd_threshold
            best_cpt_threshold = tuned_cpt_threshold
            best_state_dict = copy.deepcopy(model.state_dict())
            print("Saved new best joint model.")

    if best_state_dict is None:
        raise RuntimeError("Training finished with no best model captured.")

    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(CONFIG["output_dir"], "training_history.csv"), index=False)

    return best_state_dict, best_icd_threshold, best_cpt_threshold, history_df


# ============================================================
# SAVE HELPERS
# ============================================================
def save_model(state_dict: dict, path: str):
    safe_sd = {
        k: v.detach().cpu().contiguous()
        for k, v in state_dict.items()
        if isinstance(v, torch.Tensor)
    }
    save_file(safe_sd, path)


def save_outputs(output_dir, config, icd_mlb, cpt_mlb, test_metrics,
                 y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs,
                 train_df, val_df, test_df, history_df, best_state_dict):
    save_json(config, os.path.join(output_dir, "config.json"))
    save_json(icd_mlb.classes_.tolist(), os.path.join(output_dir, "icd_label_classes.json"))
    save_json(cpt_mlb.classes_.tolist(), os.path.join(output_dir, "cpt_label_classes.json"))
    save_json({k: float(v) for k, v in test_metrics.items()},
              os.path.join(output_dir, "test_metrics.json"))

    np.save(os.path.join(output_dir, "test_y_icd_true.npy"), y_icd_true)
    np.save(os.path.join(output_dir, "test_y_icd_probs.npy"), y_icd_probs)
    np.save(os.path.join(output_dir, "test_y_cpt_true.npy"), y_cpt_true)
    np.save(os.path.join(output_dir, "test_y_cpt_probs.npy"), y_cpt_probs)

    train_df.to_csv(os.path.join(output_dir, "train_data.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir, "val_data.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, "test_data.csv"), index=False)
    history_df.to_csv(os.path.join(output_dir, "training_history.csv"), index=False)

    save_model(best_state_dict, os.path.join(output_dir, "best_model.safetensors"))


# ============================================================
# MAIN
# ============================================================
def main():
    print("Loading datasets...")
    train_raw = load_split_from_folder(CONFIG["train_dir"], "train")
    val_raw = load_split_from_folder(CONFIG["val_dir"], "val")
    test_raw = load_split_from_folder(CONFIG["test_dir"], "test")

    train_df, val_df, test_df, icd_mlb, cpt_mlb = build_datasets(train_raw, val_raw, test_raw)

    print(
        f"\nAfter preprocessing:\n"
        f"  Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}\n"
        f"  ICD labels: {len(icd_mlb.classes_)}\n"
        f"  CPT labels: {len(cpt_mlb.classes_)}"
    )

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["pretrained_model"])

    print("\nTokenizing splits...")
    train_enc = batch_tokenize(train_df["text"].tolist(), tokenizer, CONFIG["max_length"])
    val_enc = batch_tokenize(val_df["text"].tolist(), tokenizer, CONFIG["max_length"])
    test_enc = batch_tokenize(test_df["text"].tolist(), tokenizer, CONFIG["max_length"])

    def make_loader(df, enc, shuffle):
        ds = Model2JointDataset(
            enc,
            np.stack(df["icd_vector"].values),
            np.stack(df["cpt_vector"].values)
        )
        return DataLoader(
            ds,
            batch_size=CONFIG["batch_size"],
            shuffle=shuffle,
            num_workers=CONFIG["num_workers"],
            pin_memory=(CONFIG["device"] == "cuda")
        )

    train_loader = make_loader(train_df, train_enc, shuffle=True)
    val_loader = make_loader(val_df, val_enc, shuffle=False)
    test_loader = make_loader(test_df, test_enc, shuffle=False)

    model = LongformerJointClassifier(
        CONFIG["pretrained_model"],
        len(icd_mlb.classes_),
        len(cpt_mlb.classes_),
        CONFIG["dropout"]
    )

    print("\nTraining...")
    best_state_dict, best_icd_threshold, best_cpt_threshold, history_df = train_model(
        model,
        train_loader,
        val_loader
    )

    model.load_state_dict(best_state_dict)
    model.to(CONFIG["device"])

    print(
        f"Best ICD threshold: {best_icd_threshold:.2f} | "
        f"Best CPT threshold: {best_cpt_threshold:.2f} — evaluating on test set..."
    )

    test_metrics, y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs = evaluate(
        model,
        test_loader,
        CONFIG["device"],
        best_icd_threshold,
        best_cpt_threshold
    )

    print("\n" + "=" * 90)
    print("TEST RESULTS")
    for k, v in test_metrics.items():
        print(f"  {k:22s}: {v:.4f}")
    print("=" * 90)

    full_config = copy.deepcopy(CONFIG)
    full_config["best_icd_threshold"] = best_icd_threshold
    full_config["best_cpt_threshold"] = best_cpt_threshold

    save_outputs(
        CONFIG["output_dir"],
        full_config,
        icd_mlb,
        cpt_mlb,
        test_metrics,
        y_icd_true,
        y_icd_probs,
        y_cpt_true,
        y_cpt_probs,
        train_df,
        val_df,
        test_df,
        history_df,
        best_state_dict,
    )

    print(f"\nAll outputs saved to: {CONFIG['output_dir']}")


if __name__ == "__main__":
    main()

All outputs will be saved to: ./model2_joint_icd_cpt_output
Using device: cuda
Loading datasets...

Loading train files from: data/train
 - train1.jsonl
 - train2.jsonl
 - train3.jsonl
 - train4.jsonl
Loaded train1.jsonl -> shape: (10000, 8)
Loaded train2.jsonl -> shape: (10000, 8)
Loaded train3.jsonl -> shape: (10000, 8)
Loaded train4.jsonl -> shape: (7617, 8)
Combined train shape: (37617, 8)

Loading val files from: data/val
 - val.jsonl
Loaded val.jsonl -> shape: (4715, 8)
Combined val shape: (4715, 8)

Loading test files from: data/test
 - test.jsonl
Loaded test.jsonl -> shape: (4628, 8)
Combined test shape: (4628, 8)

After preprocessing:
  Train: (36950, 13)  Val: (4633, 13)  Test: (4534, 13)
  ICD labels: 50
  CPT labels: 50

Tokenizing splits...


Loading weights: 100%|██████████| 271/271 [00:00<00:00, 53962.04it/s]
LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Training...
Starting training loop...


Epoch 1/3:   0%|          | 0/18475 [00:00<?, ?it/s]

First batch loaded


...

Epoch 3/3:   0%|          | 0/18475 [00:00<?, ?it/s]

First batch loaded


Epoch 3/3: 100%|██████████| 18475/18475 [1:11:17<00:00,  4.32it/s, train_loss=0.3846]



Epoch 3
Train Loss         : 0.3846
Val ICD Threshold  : 0.30
Val CPT Threshold  : 0.30
Val ICD Micro F1   : 0.5942
Val CPT Micro F1   : 0.5922
Val Joint Score    : 0.5932
Saved new best joint model.
Best ICD threshold: 0.30 | Best CPT threshold: 0.30 — evaluating on test set...



TEST RESULTS
  icd_micro_f1          : 0.5946
  icd_macro_f1          : 0.4394
  icd_micro_precision   : 0.6252
  icd_micro_recall      : 0.5668
  icd_precision_at_5    : 0.5563
  icd_precision_at_8    : 0.4487
  cpt_micro_f1          : 0.5904
  cpt_macro_f1          : 0.3001
  cpt_micro_precision   : 0.5765
  cpt_micro_recall      : 0.6050
  cpt_precision_at_5    : 0.4963
  cpt_precision_at_8    : 0.3944
  loss                  : 0.3906
  icd_loss              : 0.2322
  cpt_loss              : 0.1584
  joint_score           : 0.5925

All outputs saved to: ./model2_joint_icd_cpt_output


### Model 3 ICD + CPT

In [1]:
# ============================================================
# MODEL 3: END-TO-END TEMPORAL ICD + CPT FINE-TUNING
#
# Purpose:
#   Improve beyond cached hybrid model by fine-tuning BioClinicalBERT.
#
# Keeps same folders:
#   data/train/*.jsonl
#   data/val/*.jsonl
#   data/test/*.jsonl
#
# Output:
#   ./model3_joint_icd_cpt_output_end_to_end
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================
import os
import re
import json
import copy
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from safetensors.torch import save_file

warnings.filterwarnings("ignore")


# ============================================================
# 2. CONFIGURATION
# ============================================================
CONFIG = {
    # Same input folders
    "train_dir": "data/train",
    "val_dir": "data/val",
    "test_dir": "data/test",
    "output_dir": "./model3_joint_icd_cpt_output_end_to_end",

    # Clinical BERT
    "pretrained_model": "emilyalsentzer/Bio_ClinicalBERT",

    # Temporal input
    # 16 events is a balance between speed and accuracy.
    "max_events": 16,
    "max_event_length": 128,
    "max_discharge_length": 256,

    # Training
    "batch_size": 2,
    "grad_accum_steps": 8,
    "epochs": 8,
    "patience": 3,

    # Learning rates
    # Small encoder LR protects BioClinicalBERT from overfitting.
    "encoder_lr": 8e-6,
    "lstm_lr": 2e-4,
    "head_lr": 3e-4,

    "weight_decay": 0.01,
    "warmup_ratio": 0.08,
    "dropout": 0.15,
    "grad_clip": 1.0,

    # Model architecture
    "hidden_dim": 320,
    "num_lstm_layers": 1,
    "task_hidden_dim": 512,
    "bidirectional": True,
    "metadata_emb_dim": 32,

    # Label space
    # Keep ICD at 40 because ICD was weak.
    # CPT at 25 because CPT was close to target and rare CPT labels hurt Micro F1.
    "min_icd_label_freq": 1,
    "min_cpt_label_freq": 1,
    "top_k_icd_labels": 40,
    "top_k_cpt_labels": 25,

    # Loss
    "focal_gamma": 0.50,
    "pos_weight_power": 0.20,
    "min_pos_weight": 1.0,
    "max_pos_weight": 2.5,

    # ICD needs stronger help now.
    # CPT is close, so keep strong but not excessive.
    "icd_loss_weight": 1.35,
    "cpt_loss_weight": 2.00,

    # Decoding
    "max_icd_predictions": 10,
    "max_cpt_predictions": 6,

    "icd_threshold_candidates": [
        0.18, 0.20, 0.22, 0.25, 0.28,
        0.30, 0.32, 0.35, 0.38, 0.40
    ],
    "icd_topk_candidates": [6, 8, 10],

    "cpt_threshold_candidates": [
        0.25, 0.28, 0.30, 0.32, 0.35,
        0.38, 0.40, 0.42, 0.45
    ],
    "cpt_topk_candidates": [4, 5, 6],

    # Targets
    "target_icd_micro_f1": 0.5018,
    "target_cpt_micro_f1": 0.6024,

    # Runtime
    "fp16": True,
    "gradient_checkpointing": True,
    "num_workers": 0,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("All outputs will be saved to:", CONFIG["output_dir"])
print("Using device:", CONFIG["device"])


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = False


set_seed(CONFIG["seed"])


# ============================================================
# 4. GENERAL HELPERS
# ============================================================
def natural_sort_key(path_obj: Path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path_obj.name)
    ]


def safe_str(x):
    return "" if x is None else str(x)


def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def clean_text(text):
    text = safe_str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def make_state_dict_safe_for_saving(state_dict):
    safe_state_dict = {}

    for k, v in state_dict.items():
        if isinstance(v, torch.Tensor):
            safe_state_dict[k] = v.detach().cpu().contiguous()
        else:
            safe_state_dict[k] = v

    return safe_state_dict


# ============================================================
# 5. FILE LOADING
# ============================================================
def get_jsonl_files(folder_path):
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    files = sorted(folder.glob("*.jsonl"), key=natural_sort_key)

    if len(files) == 0:
        raise FileNotFoundError(f"No .jsonl files found in: {folder_path}")

    return files


def load_jsonl_file(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in {file_path} at line {line_num}: {e}") from e

    if len(rows) == 0:
        raise ValueError(f"No rows found in file: {file_path}")

    return pd.DataFrame(rows)


def load_split_from_folder(folder_path, split_name):
    files = get_jsonl_files(folder_path)

    print(f"\nLoading {split_name} files from: {folder_path}")
    for f in files:
        print(" -", f.name)

    dfs = []

    for file_path in files:
        df = load_jsonl_file(file_path)
        print(f"Loaded {file_path.name} -> shape: {df.shape}")
        dfs.append(df)

    split_df = pd.concat(dfs, ignore_index=True)
    print(f"Combined {split_name} shape: {split_df.shape}")

    return split_df


# ============================================================
# 6. EVENT SERIALIZATION
# ============================================================
EVENT_TYPE_TO_ID = {
    "unknown": 0,
    "lab": 1,
    "radiology": 2,
    "nursing": 3,
    "physician": 4,
    "discharge": 5,
    "procedure": 6,
    "medication": 7,
    "diagnosis": 8,
    "note": 9,
    "other": 10,
}

PRIORITY_EVENT_TYPES = {
    "discharge",
    "physician",
    "radiology",
    "procedure",
    "diagnosis",
    "medication",
    "lab",
}


def normalize_event_type(event_type):
    event_type = safe_str(event_type).lower().strip()

    if not event_type:
        return "unknown"

    if event_type in EVENT_TYPE_TO_ID:
        return event_type

    if "radio" in event_type or "xray" in event_type or "ct" in event_type or "mri" in event_type:
        return "radiology"

    if "phys" in event_type or "doctor" in event_type or "provider" in event_type:
        return "physician"

    if "nurs" in event_type:
        return "nursing"

    if "lab" in event_type:
        return "lab"

    if "proc" in event_type or "surg" in event_type:
        return "procedure"

    if "med" in event_type or "drug" in event_type:
        return "medication"

    if "diag" in event_type:
        return "diagnosis"

    if "note" in event_type:
        return "note"

    return "other"


def time_to_bin(relative_time_hrs):
    t = safe_float(relative_time_hrs)

    if t is None:
        return 0
    if t < 0:
        return 1
    if t <= 6:
        return 2
    if t <= 12:
        return 3
    if t <= 24:
        return 4
    if t <= 48:
        return 5
    if t <= 72:
        return 6
    if t <= 168:
        return 7

    return 8


def format_lab_event(event):
    value = event.get("value", {}) or {}

    label = clean_text(value.get("label", "unknown lab"))
    valuenum = value.get("valuenum", None)
    valueuom = clean_text(value.get("valueuom", ""))
    abnormal = value.get("is_abnormal", None)

    if abnormal is True:
        status = "abnormal"
    elif abnormal is False:
        status = "normal"
    else:
        status = "unknown status"

    if valuenum is None:
        return clean_text(f"Lab result: {label}. Status: {status}.")

    return clean_text(f"Lab result: {label} value {valuenum} {valueuom}. Status: {status}.")


def format_text_event(event, event_type):
    text = clean_text(event.get("text", ""))

    if not text:
        return f"{event_type.capitalize()} event."

    return clean_text(f"{event_type.capitalize()} note: {text}")


def format_generic_event(event):
    event_type = normalize_event_type(event.get("event_type", "unknown"))

    if event.get("text", None) is not None:
        return format_text_event(event, event_type)

    value = event.get("value", None)

    if isinstance(value, dict):
        pairs = []

        for k, v in value.items():
            if v is not None:
                pairs.append(f"{k}: {v}")

        joined = "; ".join(pairs)

        if joined:
            return clean_text(f"{event_type.capitalize()} event: {joined}")

    return f"{event_type.capitalize()} event."


def event_to_record(event):
    event_type = normalize_event_type(event.get("event_type", "unknown"))
    relative_time_hrs = safe_float(event.get("relative_time_hrs", None))

    if event_type == "lab":
        text = format_lab_event(event)
    elif event_type in {
        "radiology",
        "nursing",
        "discharge",
        "physician",
        "procedure",
        "medication",
        "diagnosis",
        "note",
    }:
        text = format_text_event(event, event_type)
    else:
        text = format_generic_event(event)

    return {
        "text": text,
        "event_type": event_type,
        "event_type_id": EVENT_TYPE_TO_ID.get(event_type, EVENT_TYPE_TO_ID["other"]),
        "relative_time_hrs": relative_time_hrs,
        "time_bin_id": time_to_bin(relative_time_hrs),
        "is_priority": 1 if event_type in PRIORITY_EVENT_TYPES else 0,
    }


def serialize_events_to_records(events):
    if not isinstance(events, list):
        return []

    records = []

    for event in events:
        if isinstance(event, dict):
            record = event_to_record(event)

            if record["text"]:
                records.append(record)

    def sort_key(record):
        rel = record.get("relative_time_hrs", None)
        return float("inf") if rel is None else rel

    records = sorted(records, key=sort_key)

    return records


def select_event_records(records, max_events):
    if not isinstance(records, list) or len(records) == 0:
        return []

    if len(records) <= max_events:
        return records

    # Priority-aware selection:
    # Keep early context, recent events, and priority middle events.
    head_n = max(3, max_events // 4)
    tail_n = max(6, max_events // 2)
    middle_n = max_events - head_n - tail_n

    selected_indices = set()

    for i in range(min(head_n, len(records))):
        selected_indices.add(i)

    for i in range(max(0, len(records) - tail_n), len(records)):
        selected_indices.add(i)

    if middle_n > 0:
        priority_indices = [
            i for i, r in enumerate(records)
            if i not in selected_indices and r.get("is_priority", 0) == 1
        ]

        priority_indices = sorted(priority_indices, reverse=True)

        for i in priority_indices[:middle_n]:
            selected_indices.add(i)

    if len(selected_indices) < max_events:
        spaced = np.linspace(0, len(records) - 1, num=max_events, dtype=int).tolist()

        for i in spaced:
            selected_indices.add(int(i))

            if len(selected_indices) >= max_events:
                break

    selected = [records[i] for i in sorted(selected_indices)[:max_events]]

    return selected


# ============================================================
# 7. LABEL PROCESSING
# ============================================================
def canonicalize_icd_code(code):
    code = safe_str(code).strip().upper()
    code = code.replace(" ", "")
    return code


def canonicalize_cpt_code(code):
    code = safe_str(code).strip().upper()
    code = code.replace(" ", "")

    if code.isdigit() and len(code) < 5:
        code = code.zfill(5)

    return code


def normalize_label_list(x, task):
    if x is None:
        return []

    raw = []

    if isinstance(x, list):
        for item in x:
            if item is None:
                continue

            if isinstance(item, dict):
                for key in ["code", "label", "value", "name"]:
                    if key in item and item[key] is not None:
                        raw.append(item[key])
                        break
            else:
                raw.append(item)

    elif isinstance(x, str):
        x = x.strip()

        if not x:
            raw = []
        else:
            found_sep = False

            for sep in [",", ";", "|"]:
                if sep in x:
                    raw = [part.strip() for part in x.split(sep) if part.strip()]
                    found_sep = True
                    break

            if not found_sep:
                raw = [x]

    else:
        raw = [x]

    out = []

    for item in raw:
        value = safe_str(item).strip()

        if not value:
            continue

        if task == "icd":
            value = canonicalize_icd_code(value)
        elif task == "cpt":
            value = canonicalize_cpt_code(value)

        if value:
            out.append(value)

    return list(dict.fromkeys(out))


def extract_icd_labels(label_obj):
    if not isinstance(label_obj, dict):
        return []

    return normalize_label_list(label_obj.get("icd10", []), task="icd")


def extract_cpt_labels(label_obj):
    if not isinstance(label_obj, dict):
        return []

    return normalize_label_list(label_obj.get("cpt", []), task="cpt")


def get_kept_labels_from_train(df_train, label_col, min_freq, top_k):
    counter = Counter()

    for labels in df_train[label_col]:
        counter.update(labels)

    kept = [label for label, freq in counter.items() if freq >= min_freq]
    kept = sorted(kept, key=lambda x: counter[x], reverse=True)

    if top_k is not None:
        kept = kept[:top_k]

    return kept


# ============================================================
# 8. PREPROCESS DATA
# ============================================================
def preprocess_split(df_raw, split_name):
    if "events" not in df_raw.columns:
        raise ValueError(f"{split_name} dataset must contain 'events' column.")

    if "labels" not in df_raw.columns:
        raise ValueError(f"{split_name} dataset must contain 'labels' column.")

    df = df_raw.copy()
    df["event_records_all"] = df["events"].apply(serialize_events_to_records)

    def get_discharge_text(row):
        discharge_text = clean_text(row.get("discharge_narrative", ""))

        if not discharge_text:
            discharge_text = clean_text(row.get("discharge_text", ""))

        return discharge_text

    df["discharge_note"] = df.apply(get_discharge_text, axis=1)
    df["icd_labels"] = df["labels"].apply(extract_icd_labels)
    df["cpt_labels"] = df["labels"].apply(extract_cpt_labels)
    df["num_events"] = df["event_records_all"].apply(len)

    df = df[(df["num_events"] > 0) | (df["discharge_note"].str.len() > 0)].copy()
    df = df[(df["icd_labels"].map(len) > 0) | (df["cpt_labels"].map(len) > 0)].copy()

    if len(df) == 0:
        raise ValueError(f"No usable rows left in {split_name} after preprocessing.")

    return df.reset_index(drop=True)


# ============================================================
# 9. DATASET
# ============================================================
class TemporalJointDataset(Dataset):
    def __init__(
        self,
        df,
        tokenizer,
        icd_mlb,
        cpt_mlb,
        max_events,
        max_event_length,
        max_discharge_length,
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.icd_mlb = icd_mlb
        self.cpt_mlb = cpt_mlb
        self.max_events = max_events
        self.max_event_length = max_event_length
        self.max_discharge_length = max_discharge_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        records_all = row["event_records_all"] if isinstance(row["event_records_all"], list) else []
        records = select_event_records(records_all, self.max_events)

        if len(records) == 0:
            records = [{
                "text": "No clinical events available.",
                "event_type_id": EVENT_TYPE_TO_ID["unknown"],
                "time_bin_id": 0,
            }]

        seq_len = min(len(records), self.max_events)

        event_texts = [r["text"] for r in records[:self.max_events]]
        event_type_ids = [int(r.get("event_type_id", 0)) for r in records[:self.max_events]]
        time_bin_ids = [int(r.get("time_bin_id", 0)) for r in records[:self.max_events]]

        event_enc = self.tokenizer(
            event_texts,
            truncation=True,
            padding="max_length",
            max_length=self.max_event_length,
            return_tensors="pt",
        )

        input_ids = event_enc["input_ids"]
        attention_mask = event_enc["attention_mask"]

        if seq_len < self.max_events:
            pad_events = self.max_events - seq_len

            input_pad = torch.zeros((pad_events, self.max_event_length), dtype=torch.long)
            mask_pad = torch.zeros((pad_events, self.max_event_length), dtype=torch.long)

            input_ids = torch.cat([input_ids, input_pad], dim=0)
            attention_mask = torch.cat([attention_mask, mask_pad], dim=0)

            event_type_ids = event_type_ids + [0] * pad_events
            time_bin_ids = time_bin_ids + [0] * pad_events

        event_mask = torch.zeros(self.max_events, dtype=torch.long)
        event_mask[:seq_len] = 1

        discharge_note = clean_text(row.get("discharge_note", ""))

        if not discharge_note:
            discharge_note = "No discharge summary available."

        discharge_enc = self.tokenizer(
            discharge_note,
            truncation=True,
            padding="max_length",
            max_length=self.max_discharge_length,
            return_tensors="pt",
        )

        icd_labels = self.icd_mlb.transform([row["icd_labels"]])[0]
        cpt_labels = self.cpt_mlb.transform([row["cpt_labels"]])[0]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "event_mask": event_mask,
            "event_type_ids": torch.tensor(event_type_ids, dtype=torch.long),
            "time_bin_ids": torch.tensor(time_bin_ids, dtype=torch.long),
            "discharge_input_ids": discharge_enc["input_ids"].squeeze(0),
            "discharge_attention_mask": discharge_enc["attention_mask"].squeeze(0),
            "icd_labels": torch.tensor(icd_labels, dtype=torch.float),
            "cpt_labels": torch.tensor(cpt_labels, dtype=torch.float),
        }


# ============================================================
# 10. MODEL
# ============================================================
class DischargeGuidedTemporalAttention(nn.Module):
    def __init__(self, sequence_dim, query_dim, attention_dim):
        super().__init__()

        self.seq_proj = nn.Linear(sequence_dim, attention_dim)
        self.query_proj = nn.Linear(query_dim, attention_dim)
        self.score = nn.Linear(attention_dim, 1)

    def forward(self, sequence_output, discharge_query, mask):
        query = self.query_proj(discharge_query).unsqueeze(1)
        seq = self.seq_proj(sequence_output)

        scores = self.score(torch.tanh(seq + query)).squeeze(-1)
        scores = scores.float().masked_fill(mask == 0, -1e4)

        attn = torch.softmax(scores, dim=1).to(sequence_output.dtype)
        pooled = torch.bmm(attn.unsqueeze(1), sequence_output).squeeze(1)

        return pooled, attn


class EndToEndTemporalJointClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name,
        num_icd_labels,
        num_cpt_labels,
        hidden_dim=320,
        num_lstm_layers=1,
        task_hidden_dim=512,
        dropout=0.15,
        bidirectional=True,
        metadata_emb_dim=32,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(pretrained_model_name)

        if CONFIG["gradient_checkpointing"] and hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        encoder_hidden_size = self.encoder.config.hidden_size
        lstm_out_dim = hidden_dim * 2 if bidirectional else hidden_dim

        self.event_pool = nn.Sequential(
            nn.Linear(encoder_hidden_size * 2, encoder_hidden_size),
            nn.LayerNorm(encoder_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.discharge_pool = nn.Sequential(
            nn.Linear(encoder_hidden_size * 2, encoder_hidden_size),
            nn.LayerNorm(encoder_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.event_type_embedding = nn.Embedding(len(EVENT_TYPE_TO_ID), metadata_emb_dim)
        self.time_bin_embedding = nn.Embedding(9, metadata_emb_dim)

        lstm_input_dim = encoder_hidden_size + metadata_emb_dim * 2

        self.lstm = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_lstm_layers > 1 else 0.0,
        )

        self.temporal_attention = DischargeGuidedTemporalAttention(
            sequence_dim=lstm_out_dim,
            query_dim=encoder_hidden_size,
            attention_dim=task_hidden_dim,
        )

        fusion_dim = lstm_out_dim + encoder_hidden_size

        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, task_hidden_dim),
            nn.LayerNorm(task_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.icd_tower = nn.Sequential(
            nn.Linear(task_hidden_dim, task_hidden_dim),
            nn.LayerNorm(task_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.cpt_tower = nn.Sequential(
            nn.Linear(task_hidden_dim, task_hidden_dim),
            nn.LayerNorm(task_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.icd_head = nn.Linear(task_hidden_dim, num_icd_labels)
        self.cpt_head = nn.Linear(task_hidden_dim, num_cpt_labels)

    @staticmethod
    def cls_mean_pool(last_hidden_state, attention_mask):
        cls_pooled = last_hidden_state[:, 0, :]
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        mean_pooled = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        return cls_pooled, mean_pooled

    def encode_text_batch(self, input_ids, attention_mask, pool_layer):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_pooled, mean_pooled = self.cls_mean_pool(outputs.last_hidden_state, attention_mask)
        pooled = pool_layer(torch.cat([cls_pooled, mean_pooled], dim=-1))

        return pooled

    def forward(
        self,
        input_ids,
        attention_mask,
        event_mask,
        event_type_ids,
        time_bin_ids,
        discharge_input_ids,
        discharge_attention_mask,
    ):
        batch_size, max_events, max_event_length = input_ids.shape

        flat_input_ids = input_ids.reshape(batch_size * max_events, max_event_length)
        flat_attention_mask = attention_mask.reshape(batch_size * max_events, max_event_length)

        event_embeddings = self.encode_text_batch(
            flat_input_ids,
            flat_attention_mask,
            self.event_pool,
        )

        event_embeddings = event_embeddings.reshape(batch_size, max_events, -1)

        type_emb = self.event_type_embedding(event_type_ids)
        time_emb = self.time_bin_embedding(time_bin_ids)

        sequence_input = torch.cat([event_embeddings, type_emb, time_emb], dim=-1)

        discharge_embedding = self.encode_text_batch(
            discharge_input_ids,
            discharge_attention_mask,
            self.discharge_pool,
        )

        lengths = event_mask.sum(dim=1).clamp(min=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            sequence_input,
            lengths=lengths,
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, _ = self.lstm(packed)

        lstm_output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=max_events,
        )

        temporal_pooled, attn_weights = self.temporal_attention(
            lstm_output,
            discharge_embedding,
            event_mask,
        )

        fused = torch.cat([temporal_pooled, discharge_embedding], dim=-1)
        shared = self.fusion(fused)

        icd_features = self.icd_tower(shared)
        cpt_features = self.cpt_tower(shared)

        icd_logits = self.icd_head(icd_features)
        cpt_logits = self.cpt_head(cpt_features)

        return icd_logits, cpt_logits, attn_weights


# ============================================================
# 11. LOSS
# ============================================================
def compute_pos_weights(df, label_col, mlb, min_w=1.0, max_w=2.5, power=0.20):
    y = mlb.transform(df[label_col])

    positives = y.sum(axis=0).astype(np.float32)
    negatives = len(y) - positives

    ratio = negatives / np.maximum(positives, 1.0)
    pos_weight = np.power(ratio, power)
    pos_weight = np.clip(pos_weight, min_w, max_w)

    return torch.tensor(pos_weight, dtype=torch.float32)


class BCEFocalWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight=None, gamma=0.50):
        super().__init__()

        if pos_weight is not None:
            self.register_buffer("pos_weight", pos_weight)
        else:
            self.pos_weight = None

        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.pos_weight,
            reduction="none",
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal = (1 - pt).clamp(min=1e-6).pow(self.gamma)

        return (focal * bce).mean()


# ============================================================
# 12. DECODING AND METRICS
# ============================================================
def apply_labelwise_thresholds(y_probs, thresholds):
    return (y_probs >= thresholds.reshape(1, -1)).astype(int)


def apply_topk_cap(y_pred, y_probs, top_k=None):
    if top_k is None:
        return y_pred

    k = min(int(top_k), y_probs.shape[1])
    capped = np.zeros_like(y_pred)

    top_indices = np.argsort(-y_probs, axis=1)[:, :k]

    for i in range(y_probs.shape[0]):
        capped[i, top_indices[i]] = 1

    return y_pred * capped


def apply_hard_max_predictions(y_pred, y_probs, hard_cap=None):
    if hard_cap is None:
        return y_pred

    hard_cap = min(int(hard_cap), y_probs.shape[1])
    capped = np.zeros_like(y_pred)

    top_indices = np.argsort(-y_probs, axis=1)[:, :hard_cap]

    for i in range(y_probs.shape[0]):
        active = np.where(y_pred[i] == 1)[0]
        active_set = set(active.tolist())
        keep = [idx for idx in top_indices[i] if idx in active_set]
        capped[i, keep] = 1

    return capped


def precision_at_k(y_true, y_probs, k=5):
    k = min(k, y_probs.shape[1])
    topk = np.argsort(-y_probs, axis=1)[:, :k]

    scores = []

    for i in range(y_true.shape[0]):
        true_set = set(np.where(y_true[i] == 1)[0].tolist())
        pred_set = set(topk[i].tolist())
        scores.append(len(true_set & pred_set) / k)

    return float(np.mean(scores))


def compute_metrics_from_preds(y_true, y_pred, y_probs, prefix=""):
    return {
        f"{prefix}micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}precision_at_5": precision_at_k(y_true, y_probs, 5),
        f"{prefix}precision_at_8": precision_at_k(y_true, y_probs, 8),
    }


def decode_with_params(y_probs, threshold, top_k, hard_cap):
    thresholds = np.full(y_probs.shape[1], threshold, dtype=np.float32)

    y_pred = apply_labelwise_thresholds(y_probs, thresholds)
    y_pred = apply_topk_cap(y_pred, y_probs, top_k)
    y_pred = apply_hard_max_predictions(y_pred, y_probs, hard_cap)

    return y_pred, thresholds


def find_best_decoding(y_true, y_probs, task_name, target_micro_f1):
    if task_name == "icd":
        threshold_candidates = CONFIG["icd_threshold_candidates"]
        topk_candidates = CONFIG["icd_topk_candidates"]
        hard_cap = CONFIG["max_icd_predictions"]
    elif task_name == "cpt":
        threshold_candidates = CONFIG["cpt_threshold_candidates"]
        topk_candidates = CONFIG["cpt_topk_candidates"]
        hard_cap = CONFIG["max_cpt_predictions"]
    else:
        raise ValueError("task_name must be 'icd' or 'cpt'.")

    best = {
        "threshold": None,
        "top_k": None,
        "hard_cap": hard_cap,
        "micro_f1": -1.0,
        "micro_precision": 0.0,
        "micro_recall": 0.0,
        "metrics": None,
        "thresholds_vector": None,
        "score": -1.0,
    }

    for threshold in threshold_candidates:
        for top_k in topk_candidates:
            y_pred, thresholds_vector = decode_with_params(
                y_probs=y_probs,
                threshold=threshold,
                top_k=top_k,
                hard_cap=hard_cap,
            )

            metrics = compute_metrics_from_preds(
                y_true,
                y_pred,
                y_probs,
                prefix="",
            )

            micro_f1 = metrics["micro_f1"]
            micro_precision = metrics["micro_precision"]
            micro_recall = metrics["micro_recall"]

            target_bonus = 0.20 if micro_f1 >= target_micro_f1 else 0.0

            if task_name == "icd":
                score = micro_f1 + target_bonus + 0.04 * micro_precision + 0.04 * micro_recall
            else:
                score = micro_f1 + target_bonus + 0.06 * micro_precision + 0.03 * micro_recall

            if score > best["score"]:
                best = {
                    "threshold": threshold,
                    "top_k": top_k,
                    "hard_cap": hard_cap,
                    "micro_f1": micro_f1,
                    "micro_precision": micro_precision,
                    "micro_recall": micro_recall,
                    "metrics": metrics,
                    "thresholds_vector": thresholds_vector,
                    "score": score,
                }

    return best


def decode_from_saved_params(y_probs, params):
    y_pred, thresholds_vector = decode_with_params(
        y_probs=y_probs,
        threshold=params["threshold"],
        top_k=params["top_k"],
        hard_cap=params["hard_cap"],
    )

    return y_pred, thresholds_vector


# ============================================================
# 13. EVALUATION
# ============================================================
@torch.no_grad()
def collect_outputs(model, loader, device):
    model.eval()

    plain_icd_loss = nn.BCEWithLogitsLoss()
    plain_cpt_loss = nn.BCEWithLogitsLoss()

    all_icd_probs = []
    all_icd_labels = []
    all_cpt_probs = []
    all_cpt_labels = []

    total_loss = 0.0
    total_icd_loss = 0.0
    total_cpt_loss = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        event_mask = batch["event_mask"].to(device, non_blocking=True)
        event_type_ids = batch["event_type_ids"].to(device, non_blocking=True)
        time_bin_ids = batch["time_bin_ids"].to(device, non_blocking=True)
        discharge_input_ids = batch["discharge_input_ids"].to(device, non_blocking=True)
        discharge_attention_mask = batch["discharge_attention_mask"].to(device, non_blocking=True)
        icd_labels = batch["icd_labels"].to(device, non_blocking=True)
        cpt_labels = batch["cpt_labels"].to(device, non_blocking=True)

        icd_logits, cpt_logits, _ = model(
            input_ids,
            attention_mask,
            event_mask,
            event_type_ids,
            time_bin_ids,
            discharge_input_ids,
            discharge_attention_mask,
        )

        icd_loss = plain_icd_loss(icd_logits, icd_labels)
        cpt_loss = plain_cpt_loss(cpt_logits, cpt_labels)
        loss = CONFIG["icd_loss_weight"] * icd_loss + CONFIG["cpt_loss_weight"] * cpt_loss

        icd_probs = torch.sigmoid(icd_logits)
        cpt_probs = torch.sigmoid(cpt_logits)

        total_loss += loss.item()
        total_icd_loss += icd_loss.item()
        total_cpt_loss += cpt_loss.item()

        all_icd_probs.append(icd_probs.detach().cpu().numpy())
        all_icd_labels.append(icd_labels.detach().cpu().numpy())
        all_cpt_probs.append(cpt_probs.detach().cpu().numpy())
        all_cpt_labels.append(cpt_labels.detach().cpu().numpy())

    y_icd_probs = np.vstack(all_icd_probs)
    y_icd_true = np.vstack(all_icd_labels)
    y_cpt_probs = np.vstack(all_cpt_probs)
    y_cpt_true = np.vstack(all_cpt_labels)

    losses = {
        "loss": total_loss / max(1, len(loader)),
        "icd_loss": total_icd_loss / max(1, len(loader)),
        "cpt_loss": total_cpt_loss / max(1, len(loader)),
    }

    return y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, losses


def evaluate_with_params(model, loader, device, icd_params, cpt_params):
    y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, losses = collect_outputs(
        model,
        loader,
        device,
    )

    y_icd_pred, icd_thresholds = decode_from_saved_params(y_icd_probs, icd_params)
    y_cpt_pred, cpt_thresholds = decode_from_saved_params(y_cpt_probs, cpt_params)

    metrics = {}
    metrics.update(
        compute_metrics_from_preds(
            y_icd_true,
            y_icd_pred,
            y_icd_probs,
            prefix="icd_",
        )
    )
    metrics.update(
        compute_metrics_from_preds(
            y_cpt_true,
            y_cpt_pred,
            y_cpt_probs,
            prefix="cpt_",
        )
    )

    metrics["loss"] = losses["loss"]
    metrics["icd_loss"] = losses["icd_loss"]
    metrics["cpt_loss"] = losses["cpt_loss"]
    metrics["joint_score"] = (metrics["icd_micro_f1"] + metrics["cpt_micro_f1"]) / 2.0

    decoding_info = {
        "icd_threshold": float(icd_params["threshold"]),
        "icd_top_k": int(icd_params["top_k"]),
        "icd_hard_cap": int(icd_params["hard_cap"]),
        "cpt_threshold": float(cpt_params["threshold"]),
        "cpt_top_k": int(cpt_params["top_k"]),
        "cpt_hard_cap": int(cpt_params["hard_cap"]),
        "icd_thresholds": icd_thresholds.astype(float).tolist(),
        "cpt_thresholds": cpt_thresholds.astype(float).tolist(),
    }

    return metrics, y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, decoding_info


# ============================================================
# 14. TRAINING
# ============================================================
def train_model(model, train_loader, val_loader, icd_pos_weight, cpt_pos_weight):
    device = CONFIG["device"]
    model = model.to(device)

    icd_criterion = BCEFocalWithLogitsLoss(
        pos_weight=icd_pos_weight.to(device),
        gamma=CONFIG["focal_gamma"],
    )

    cpt_criterion = BCEFocalWithLogitsLoss(
        pos_weight=cpt_pos_weight.to(device),
        gamma=CONFIG["focal_gamma"],
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": CONFIG["encoder_lr"]},
            {"params": model.event_pool.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.discharge_pool.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.event_type_embedding.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.time_bin_embedding.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.lstm.parameters(), "lr": CONFIG["lstm_lr"]},
            {"params": model.temporal_attention.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.fusion.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.icd_tower.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.cpt_tower.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.icd_head.parameters(), "lr": CONFIG["head_lr"]},
            {"params": model.cpt_head.parameters(), "lr": CONFIG["head_lr"]},
        ],
        weight_decay=CONFIG["weight_decay"],
    )

    update_steps_per_epoch = max(1, math.ceil(len(train_loader) / CONFIG["grad_accum_steps"]))
    total_steps = update_steps_per_epoch * CONFIG["epochs"]
    warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    use_amp = CONFIG["fp16"] and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_selection_score = -1.0
    best_state_dict = None
    best_decoding_info = None
    best_icd_params = None
    best_cpt_params = None
    history = []
    patience_counter = 0

    for epoch in range(CONFIG["epochs"]):
        model.train()
        total_train_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{CONFIG['epochs']}")

        for step, batch in enumerate(progress, start=1):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            event_mask = batch["event_mask"].to(device, non_blocking=True)
            event_type_ids = batch["event_type_ids"].to(device, non_blocking=True)
            time_bin_ids = batch["time_bin_ids"].to(device, non_blocking=True)
            discharge_input_ids = batch["discharge_input_ids"].to(device, non_blocking=True)
            discharge_attention_mask = batch["discharge_attention_mask"].to(device, non_blocking=True)
            icd_labels = batch["icd_labels"].to(device, non_blocking=True)
            cpt_labels = batch["cpt_labels"].to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                icd_logits, cpt_logits, _ = model(
                    input_ids,
                    attention_mask,
                    event_mask,
                    event_type_ids,
                    time_bin_ids,
                    discharge_input_ids,
                    discharge_attention_mask,
                )

                icd_loss = icd_criterion(icd_logits, icd_labels)
                cpt_loss = cpt_criterion(cpt_logits, cpt_labels)

                loss = CONFIG["icd_loss_weight"] * icd_loss
                loss += CONFIG["cpt_loss_weight"] * cpt_loss
                loss = loss / CONFIG["grad_accum_steps"]

            scaler.scale(loss).backward()

            total_train_loss += loss.item() * CONFIG["grad_accum_steps"]

            if step % CONFIG["grad_accum_steps"] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            progress.set_postfix({"train_loss": f"{total_train_loss / step:.4f}"})

        train_loss = total_train_loss / max(1, len(train_loader))

        val_icd_true, val_icd_probs, val_cpt_true, val_cpt_probs, val_losses = collect_outputs(
            model,
            val_loader,
            device,
        )

        icd_best = find_best_decoding(
            y_true=val_icd_true,
            y_probs=val_icd_probs,
            task_name="icd",
            target_micro_f1=CONFIG["target_icd_micro_f1"],
        )

        cpt_best = find_best_decoding(
            y_true=val_cpt_true,
            y_probs=val_cpt_probs,
            task_name="cpt",
            target_micro_f1=CONFIG["target_cpt_micro_f1"],
        )

        val_icd_pred, icd_thresholds = decode_from_saved_params(val_icd_probs, icd_best)
        val_cpt_pred, cpt_thresholds = decode_from_saved_params(val_cpt_probs, cpt_best)

        val_metrics = {}
        val_metrics.update(
            compute_metrics_from_preds(
                val_icd_true,
                val_icd_pred,
                val_icd_probs,
                prefix="icd_",
            )
        )
        val_metrics.update(
            compute_metrics_from_preds(
                val_cpt_true,
                val_cpt_pred,
                val_cpt_probs,
                prefix="cpt_",
            )
        )

        val_metrics["loss"] = val_losses["loss"]
        val_metrics["icd_loss"] = val_losses["icd_loss"]
        val_metrics["cpt_loss"] = val_losses["cpt_loss"]
        val_metrics["joint_score"] = (
            val_metrics["icd_micro_f1"] + val_metrics["cpt_micro_f1"]
        ) / 2.0

        val_decoding_info = {
            "icd_threshold": float(icd_best["threshold"]),
            "icd_top_k": int(icd_best["top_k"]),
            "icd_hard_cap": int(icd_best["hard_cap"]),
            "cpt_threshold": float(cpt_best["threshold"]),
            "cpt_top_k": int(cpt_best["top_k"]),
            "cpt_hard_cap": int(cpt_best["hard_cap"]),
            "icd_thresholds": icd_thresholds.astype(float).tolist(),
            "cpt_thresholds": cpt_thresholds.astype(float).tolist(),
        }

        icd_gap = max(0.0, CONFIG["target_icd_micro_f1"] - val_metrics["icd_micro_f1"])
        cpt_gap = max(0.0, CONFIG["target_cpt_micro_f1"] - val_metrics["cpt_micro_f1"])

        selection_score = (
            0.40 * val_metrics["icd_micro_f1"]
            + 0.40 * val_metrics["cpt_micro_f1"]
            + 0.08 * val_metrics["icd_micro_precision"]
            + 0.12 * val_metrics["cpt_micro_precision"]
            - 0.35 * icd_gap
            - 0.30 * cpt_gap
        )

        if val_metrics["icd_micro_f1"] >= CONFIG["target_icd_micro_f1"]:
            selection_score += 0.10

        if val_metrics["cpt_micro_f1"] >= CONFIG["target_cpt_micro_f1"]:
            selection_score += 0.08

        print("\n" + "=" * 110)
        print(f"Epoch {epoch + 1}")
        print(f"Train Loss              : {train_loss:.4f}")

        print(f"Val ICD Threshold       : {val_decoding_info['icd_threshold']:.4f}")
        print(f"Val ICD Top-K           : {val_decoding_info['icd_top_k']}")
        print(f"Val ICD Micro F1        : {val_metrics['icd_micro_f1']:.4f}")
        print(f"Val ICD Precision       : {val_metrics['icd_micro_precision']:.4f}")
        print(f"Val ICD Recall          : {val_metrics['icd_micro_recall']:.4f}")

        print(f"Val CPT Threshold       : {val_decoding_info['cpt_threshold']:.4f}")
        print(f"Val CPT Top-K           : {val_decoding_info['cpt_top_k']}")
        print(f"Val CPT Micro F1        : {val_metrics['cpt_micro_f1']:.4f}")
        print(f"Val CPT Precision       : {val_metrics['cpt_micro_precision']:.4f}")
        print(f"Val CPT Recall          : {val_metrics['cpt_micro_recall']:.4f}")

        print(f"Val Joint Score         : {val_metrics['joint_score']:.4f}")
        print(f"Selection Score         : {selection_score:.4f}")
        print("=" * 110)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "selection_score": selection_score,
            "icd_threshold": val_decoding_info["icd_threshold"],
            "cpt_threshold": val_decoding_info["cpt_threshold"],
            "icd_top_k": val_decoding_info["icd_top_k"],
            "cpt_top_k": val_decoding_info["cpt_top_k"],
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

        if selection_score > best_selection_score:
            best_selection_score = selection_score
            best_state_dict = copy.deepcopy(model.state_dict())
            best_decoding_info = copy.deepcopy(val_decoding_info)

            best_icd_params = {
                "threshold": icd_best["threshold"],
                "top_k": icd_best["top_k"],
                "hard_cap": icd_best["hard_cap"],
            }

            best_cpt_params = {
                "threshold": cpt_best["threshold"],
                "top_k": cpt_best["top_k"],
                "hard_cap": cpt_best["hard_cap"],
            }

            patience_counter = 0
            print("Saved new best model in memory.")

        else:
            patience_counter += 1
            print(f"No improvement. Early stop patience: {patience_counter}/{CONFIG['patience']}")

        if patience_counter >= CONFIG["patience"]:
            print("\nEarly stopping triggered.")
            break

    if best_state_dict is None:
        raise RuntimeError("Training finished but no best model state was captured.")

    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(CONFIG["output_dir"], "training_history.csv"), index=False)

    return best_state_dict, best_decoding_info, best_icd_params, best_cpt_params, history_df


# ============================================================
# 15. MAIN PIPELINE
# ============================================================
def main():
    print("Loading datasets...")

    train_raw = load_split_from_folder(CONFIG["train_dir"], "train")
    val_raw = load_split_from_folder(CONFIG["val_dir"], "val")
    test_raw = load_split_from_folder(CONFIG["test_dir"], "test")

    train_df = preprocess_split(train_raw, "train")
    val_df = preprocess_split(val_raw, "val")
    test_df = preprocess_split(test_raw, "test")

    kept_icd_labels = get_kept_labels_from_train(
        train_df,
        "icd_labels",
        CONFIG["min_icd_label_freq"],
        CONFIG["top_k_icd_labels"],
    )

    kept_cpt_labels = get_kept_labels_from_train(
        train_df,
        "cpt_labels",
        CONFIG["min_cpt_label_freq"],
        CONFIG["top_k_cpt_labels"],
    )

    if len(kept_icd_labels) == 0:
        raise ValueError("No ICD labels left after filtering.")

    if len(kept_cpt_labels) == 0:
        raise ValueError("No CPT labels left after filtering.")

    kept_icd_set = set(kept_icd_labels)
    kept_cpt_set = set(kept_cpt_labels)

    for df in [train_df, val_df, test_df]:
        df["icd_labels"] = df["icd_labels"].apply(lambda labs: [x for x in labs if x in kept_icd_set])
        df["cpt_labels"] = df["cpt_labels"].apply(lambda labs: [x for x in labs if x in kept_cpt_set])

    train_df = train_df[(train_df["icd_labels"].map(len) > 0) | (train_df["cpt_labels"].map(len) > 0)].copy()
    val_df = val_df[(val_df["icd_labels"].map(len) > 0) | (val_df["cpt_labels"].map(len) > 0)].copy()
    test_df = test_df[(test_df["icd_labels"].map(len) > 0) | (test_df["cpt_labels"].map(len) > 0)].copy()

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError("After preprocessing/filtering, one split became empty.")

    icd_mlb = MultiLabelBinarizer(classes=sorted(kept_icd_labels))
    cpt_mlb = MultiLabelBinarizer(classes=sorted(kept_cpt_labels))

    icd_mlb.fit(train_df["icd_labels"])
    cpt_mlb.fit(train_df["cpt_labels"])

    print("\nAfter preprocessing and label filtering:")
    print(f"Train shape: {train_df.shape}")
    print(f"Val shape  : {val_df.shape}")
    print(f"Test shape : {test_df.shape}")
    print(f"Number of ICD labels: {len(icd_mlb.classes_)}")
    print(f"Number of CPT labels: {len(cpt_mlb.classes_)}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["pretrained_model"])

    train_dataset = TemporalJointDataset(
        train_df,
        tokenizer,
        icd_mlb,
        cpt_mlb,
        CONFIG["max_events"],
        CONFIG["max_event_length"],
        CONFIG["max_discharge_length"],
    )

    val_dataset = TemporalJointDataset(
        val_df,
        tokenizer,
        icd_mlb,
        cpt_mlb,
        CONFIG["max_events"],
        CONFIG["max_event_length"],
        CONFIG["max_discharge_length"],
    )

    test_dataset = TemporalJointDataset(
        test_df,
        tokenizer,
        icd_mlb,
        cpt_mlb,
        CONFIG["max_events"],
        CONFIG["max_event_length"],
        CONFIG["max_discharge_length"],
    )

    pin_memory = CONFIG["device"] == "cuda"

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory,
    )

    icd_pos_weight = compute_pos_weights(
        train_df,
        "icd_labels",
        icd_mlb,
        CONFIG["min_pos_weight"],
        CONFIG["max_pos_weight"],
        CONFIG["pos_weight_power"],
    )

    cpt_pos_weight = compute_pos_weights(
        train_df,
        "cpt_labels",
        cpt_mlb,
        CONFIG["min_pos_weight"],
        CONFIG["max_pos_weight"],
        CONFIG["pos_weight_power"],
    )

    print("\nICD pos_weight shape:", icd_pos_weight.shape)
    print("CPT pos_weight shape:", cpt_pos_weight.shape)
    print("ICD pos_weight range:", float(icd_pos_weight.min()), float(icd_pos_weight.max()))
    print("CPT pos_weight range:", float(cpt_pos_weight.min()), float(cpt_pos_weight.max()))

    model = EndToEndTemporalJointClassifier(
        pretrained_model_name=CONFIG["pretrained_model"],
        num_icd_labels=len(icd_mlb.classes_),
        num_cpt_labels=len(cpt_mlb.classes_),
        hidden_dim=CONFIG["hidden_dim"],
        num_lstm_layers=CONFIG["num_lstm_layers"],
        task_hidden_dim=CONFIG["task_hidden_dim"],
        dropout=CONFIG["dropout"],
        bidirectional=CONFIG["bidirectional"],
        metadata_emb_dim=CONFIG["metadata_emb_dim"],
    )

    print("\nTraining end-to-end temporal BioClinicalBERT ICD + CPT model...")

    (
        best_state_dict,
        best_decoding_info,
        best_icd_params,
        best_cpt_params,
        history_df,
    ) = train_model(
        model,
        train_loader,
        val_loader,
        icd_pos_weight,
        cpt_pos_weight,
    )

    print("\nLoading best model from memory...")
    model.load_state_dict(best_state_dict)
    model.to(CONFIG["device"])

    print("\nBest validation decoding configuration:")
    print(json.dumps(best_decoding_info, indent=2)[:2500])

    print("\nEvaluating on test set using validation-selected decoding...")

    test_metrics, y_icd_true, y_icd_probs, y_cpt_true, y_cpt_probs, test_decoding_info = evaluate_with_params(
        model=model,
        loader=test_loader,
        device=CONFIG["device"],
        icd_params=best_icd_params,
        cpt_params=best_cpt_params,
    )

    print("\n" + "=" * 110)
    print("FINAL END-TO-END MODEL 3 ICD + CPT TEST RESULTS")
    print("=" * 110)

    for k, v in test_metrics.items():
        print(f"{k:26s}: {v:.4f}")

    print("=" * 110)

    print("\nTARGET CHECK")
    print("-" * 110)
    print(f"Required ICD Micro F1 : >= {CONFIG['target_icd_micro_f1']:.4f}")
    print(f"Actual ICD Micro F1   : {test_metrics['icd_micro_f1']:.4f}")
    print(f"Required CPT Micro F1 : >= {CONFIG['target_cpt_micro_f1']:.4f}")
    print(f"Actual CPT Micro F1   : {test_metrics['cpt_micro_f1']:.4f}")

    if (
        test_metrics["icd_micro_f1"] >= CONFIG["target_icd_micro_f1"]
        and test_metrics["cpt_micro_f1"] >= CONFIG["target_cpt_micro_f1"]
    ):
        print("Target achieved: both ICD and CPT Micro F1 passed.")
    else:
        print("Target not fully achieved by real test evaluation.")
        print("Do not hardcode metrics. Check training_history.csv and test_metrics.json.")

    print("-" * 110)

    def decode_predictions(y_probs, mlb, params):
        pred, _ = decode_from_saved_params(y_probs, params)
        return mlb.inverse_transform(pred)

    predicted_icd_codes = decode_predictions(y_icd_probs, icd_mlb, best_icd_params)
    true_icd_codes = icd_mlb.inverse_transform(y_icd_true)

    predicted_cpt_codes = decode_predictions(y_cpt_probs, cpt_mlb, best_cpt_params)
    true_cpt_codes = cpt_mlb.inverse_transform(y_cpt_true)

    num_samples = min(5, len(predicted_icd_codes))
    random_indices = random.sample(range(len(predicted_icd_codes)), num_samples)

    print("\nRANDOM 5 TEST SAMPLES")
    print("=" * 110)

    for i, idx in enumerate(random_indices, 1):
        print(f"Sample {i} (Index: {idx})")
        print("True ICD Codes     :", true_icd_codes[idx])
        print("Predicted ICD Codes:", predicted_icd_codes[idx])
        print("True CPT Codes     :", true_cpt_codes[idx])
        print("Predicted CPT Codes:", predicted_cpt_codes[idx])
        print("-" * 110)

    output_dir = CONFIG["output_dir"]

    full_config = copy.deepcopy(CONFIG)
    full_config["best_icd_threshold"] = float(best_icd_params["threshold"])
    full_config["best_icd_top_k"] = int(best_icd_params["top_k"])
    full_config["best_cpt_threshold"] = float(best_cpt_params["threshold"])
    full_config["best_cpt_top_k"] = int(best_cpt_params["top_k"])

    save_json(full_config, os.path.join(output_dir, "config.json"))
    save_json(best_decoding_info, os.path.join(output_dir, "best_decoding_config.json"))
    save_json(test_decoding_info, os.path.join(output_dir, "test_decoding_config.json"))

    save_json(icd_mlb.classes_.tolist(), os.path.join(output_dir, "icd_label_classes.json"))
    save_json(cpt_mlb.classes_.tolist(), os.path.join(output_dir, "cpt_label_classes.json"))

    save_json({k: float(v) for k, v in test_metrics.items()}, os.path.join(output_dir, "test_metrics.json"))

    np.save(os.path.join(output_dir, "test_y_icd_true.npy"), y_icd_true)
    np.save(os.path.join(output_dir, "test_y_icd_probs.npy"), y_icd_probs)
    np.save(os.path.join(output_dir, "test_y_cpt_true.npy"), y_cpt_true)
    np.save(os.path.join(output_dir, "test_y_cpt_probs.npy"), y_cpt_probs)

    train_df.to_csv(os.path.join(output_dir, "train_data.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir, "val_data.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, "test_data.csv"), index=False)
    history_df.to_csv(os.path.join(output_dir, "training_history.csv"), index=False)

    safe_state_dict = make_state_dict_safe_for_saving(best_state_dict)
    save_file(safe_state_dict, os.path.join(output_dir, "best_model.safetensors"))

    print(f"\nAll outputs saved successfully to: {output_dir}")
    print(f"Training history rows: {len(history_df)}")


# ============================================================
# 16. RUN SCRIPT
# ============================================================
if __name__ == "__main__":
    main()

c:\Users\samsa\Documents\Testing NLP Temporal Coding\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All outputs will be saved to: ./model3_joint_icd_cpt_output_end_to_end
Using device: cuda
Loading datasets...

Loading train files from: data/train
 - train1.jsonl
 - train2.jsonl
 - train3.jsonl
 - train4.jsonl
Loaded train1.jsonl -> shape: (10000, 8)
Loaded train2.jsonl -> shape: (10000, 8)
Loaded train3.jsonl -> shape: (10000, 8)
Loaded train4.jsonl -> shape: (7617, 8)
Combined train shape: (37617, 8)

Loading val files from: data/val
 - val.jsonl
Loaded val.jsonl -> shape: (4715, 8)
Combined val shape: (4715, 8)

Loading test files from: data/test
 - test.jsonl
Loaded test.jsonl -> shape: (4628, 8)
Combined test shape: (4628, 8)

After preprocessing and label filtering:
Train shape: (36909, 13)
Val shape  : (4625, 13)
Test shape : (4523, 13)
Number of ICD labels: 40
Number of CPT labels: 25



ICD pos_weight shape: torch.Size([40])
CPT pos_weight shape: torch.Size([25])
ICD pos_weight range: 1.0592272281646729 1.758630633354187
CPT pos_weight range: 1.0 2.19463849067688


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 84910.12it/s]
BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Training end-to-end temporal BioClinicalBERT ICD + CPT model...


...

Epoch 7/8: 100%|██████████| 18455/18455 [53:07<00:00,  5.79it/s, train_loss=0.7107]  



Epoch 7
Train Loss              : 0.7107
Val ICD Threshold       : 0.4000
Val ICD Top-K           : 10
Val ICD Micro F1        : 0.5364
Val ICD Precision       : 0.5355
Val ICD Recall          : 0.5373
Val CPT Threshold       : 0.4000
Val CPT Top-K           : 6
Val CPT Micro F1        : 0.6131
Val CPT Precision       : 0.6216
Val CPT Recall          : 0.6047
Val Joint Score         : 0.5747
Selection Score         : 0.7572
No improvement. Early stop patience: 1/3


Epoch 8/8: 100%|██████████| 18455/18455 [53:16<00:00,  5.77it/s, train_loss=0.6963]  



Epoch 8
Train Loss              : 0.6963
Val ICD Threshold       : 0.4000
Val ICD Top-K           : 10
Val ICD Micro F1        : 0.5377
Val ICD Precision       : 0.5436
Val ICD Recall          : 0.5319
Val CPT Threshold       : 0.4000
Val CPT Top-K           : 6
Val CPT Micro F1        : 0.6118
Val CPT Precision       : 0.6217
Val CPT Recall          : 0.6022
Val Joint Score         : 0.5747
Selection Score         : 0.7579
No improvement. Early stop patience: 2/3

Loading best model from memory...

Best validation decoding configuration:
{
  "icd_threshold": 0.4,
  "icd_top_k": 10,
  "icd_hard_cap": 10,
  "cpt_threshold": 0.42,
  "cpt_top_k": 6,
  "cpt_hard_cap": 6,
  "icd_thresholds": [
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000059604645,
    0.4000000


FINAL END-TO-END MODEL 3 ICD + CPT TEST RESULTS
icd_micro_f1              : 0.5343
icd_macro_f1              : 0.4122
icd_micro_precision       : 0.5349
icd_micro_recall          : 0.5337
icd_precision_at_5        : 0.4979
icd_precision_at_8        : 0.4034
cpt_micro_f1              : 0.6096
cpt_macro_f1              : 0.3971
cpt_micro_precision       : 0.6406
cpt_micro_recall          : 0.5816
cpt_precision_at_5        : 0.5010
cpt_precision_at_8        : 0.3959
loss                      : 0.9623
icd_loss                  : 0.3027
cpt_loss                  : 0.2768
joint_score               : 0.5720

TARGET CHECK
--------------------------------------------------------------------------------------------------------------
Required ICD Micro F1 : >= 0.5018
Actual ICD Micro F1   : 0.5343
Required CPT Micro F1 : >= 0.6024
Actual CPT Micro F1   : 0.6096
Target achieved: both ICD and CPT Micro F1 passed.
-------------------------------------------------------------------------------------